In [1]:
import pandas as pd
import datetime
# import pymysql
import pandas.io.sql as psql
from datetime import datetime as dt
import numpy as np
import pandas.tseries.offsets as offsets
import python_ss.python_ss as ps
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import json

import os
import ast
import db_dtypes
from google.cloud import bigquery
from google.oauth2 import service_account
from google.cloud import secretmanager
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from google_auth_oauthlib.flow import InstalledAppFlow
# importでエラーが出てしまった場合は、コマンドプロンプトにて「pip install ”必要なモジュール”」でインストールしていただく必要がございます。
# 例. pip install db_dtypes

import xlsxwriter
import re

import os
from decimal import Decimal
import calendar
# import utils
# from utils import *
#importlib.reload(utils)
print(os.getcwd())



z:\Users\suehara\Documents\python\analysis\yojitu


In [2]:
pd.options.display.max_rows = 10000
pd.options.display.max_columns = 200


In [3]:
#転機IDの10000以降の手上げ情報取得
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly',
          'https://www.googleapis.com/auth/spreadsheets']
json_path = r"Z:\Users\suehara\Documents\python\analysis\yojitu\python_ss\credentials.json"

#カレンダー用の週次まとめを行う
service = ps.get_auth(SCOPES,json_path)
SPREADSHEET_ID = '11scU7ixGvt2JYSBQlHkGMKZSLSzYbrmG221CXUGDqDU'
Sheet_NAME = 'masta!'
Sheet_row = "A:D"
RANGE_NAME = Sheet_NAME+Sheet_row
master1 = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)

# ロンザンメンバーの在籍・退職異動・推移を見る
Sheet_row = "G:M"
RANGE_NAME = Sheet_NAME+Sheet_row
master2 = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)

# ヨミ表のAPソースを統一する
Sheet_row = "O:P"
RANGE_NAME = Sheet_NAME+Sheet_row
master3 = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)

# 顧客支持ポイントの掛率を統一する
Sheet_row = "R:U"
RANGE_NAME = Sheet_NAME+Sheet_row
master4 = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)

# 日付にQや同営業日の情報を加える
Sheet_NAME = 'Q営業日!'
Sheet_row = "A:O"
RANGE_NAME = Sheet_NAME+Sheet_row
Q_master = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)
Q_master['月'] = pd.to_datetime(Q_master['月'], errors='coerce')
Q_master['日付'] = pd.to_datetime(Q_master['日付'])


In [ ]:
def aggregate_monthly_trends(df_transactions: pd.DataFrame, Q_master: pd.DataFrame) -> pd.DataFrame:
    """
    トランザクションデータとQマスタを結合し、Qおよび初中最終月ごとの推移を集計します。
    """
    # 1. Qマスタのメモリ最適化と不要カラムの除外
    # 結合に必要なカラムのみに絞り、結合時のメモリ消費を最小化
    df_q = Q_master[['日付', 'Q', '初中最終月']].copy()
    
    # 日付型への変換（すでに変換済みの場合は不要）
    df_q['日付'] = pd.to_datetime(df_q['日付'])
    
    # 【重要】順序付きカテゴリ型への変換
    # メモリを削減し、集計時のソート順（初月 -> 中月 -> 最終月）を保証する
    months_order = ['初月', '中月', '最終月']
    df_q['初中最終月'] = pd.Categorical(df_q['初中最終月'], categories=months_order, ordered=True)
    
    # 2. トランザクションデータの準備
    # トランザクション側の日付も datetime 型に揃える
    df_transactions['action_date'] = pd.to_datetime(df_transactions['action_date'])
    
    # 3. 結合 (Merge)
    # 日付をキーにして実績データにQと初中最終月を付与
    merged_df = pd.merge(
        df_transactions, 
        df_q, 
        left_on='action_date', 
        right_on='日付', 
        how='left'
    )
    
    # 4. 集計 (Group By)
    # observed=True を指定することで、データが存在しないカテゴリの組み合わせを無視し高速化
    # 例として 'sales' カラムや 'id' のカウント等、目的に応じて変更してください
    summary_df = (
        merged_df.groupby(['Q', '初中最終月'], observed=True)
        .agg(
            action_count=('id', 'count'),  # レコード数のカウント
            # total_sales=('sales', 'sum') # 売上合計などが必要な場合は追加
        )
        .reset_index()
    )
    
    # Q順、初中最終月順にソート（Categorical型のおかげで自然にソートされます）
    summary_df = summary_df.sort_values(['Q', '初中最終月'])
    
    return summary_df

In [4]:
#master3 = master3.rename(columns={"人マスタ.1": "人マスタ","sei_plus.1":"sei_plus"}) #カラム名変更
#master3 = master3.dropna(subset=['人マスタ', 'sei_plus'])

master1['日付'] = pd.to_datetime(master1['日付']) 

master2 = master2.dropna(subset=['人マスタ', 'sei_plus'])

master3 = master3.dropna(subset=['ヨミ表選択'])

master4 = master4.dropna(subset=['計上Q'])

In [5]:
# Bigqueryを使えるようにするためのおまじない
def access_secret_version(project_id, secret_id, version_id='latest'):
    client = secretmanager.SecretManagerServiceClient()

    name = f"projects/{project_id}/secrets/{secret_id}/versions/{version_id}"
    response = client.access_secret_version(request={"name": name})
    payload = response.payload.data.decode("UTF-8")
    return ast.literal_eval(payload)

# 上記関数を実行するコードが記載されています。こちらもそのままお使いください。
credentials = service_account.Credentials.from_service_account_info(
access_secret_version('r-group-bigdata', 'CREDENTIALS_SECRET_KEY_WORKER'),
scopes=["https://www.googleapis.com/auth/cloud-platform"],)


z:\Users\suehara\Documents\python\analysis\.venv\Lib\site-packages\google\auth\_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [6]:
# 当Qの開始日と最終日を取得する
sql = """
SELECT
  CONCAT(period,"-",quarter,"Q") AS Q,
  MIN(date) AS first_date,
  MAX(date) AS end_date
FROM `r-group-bigdata.koyomi.calendar`
GROUP BY CONCAT(period,"-",quarter,"Q")
ORDER BY first_date
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
date_df = client.query(sql).result().to_dataframe()

today = pd.to_datetime(dt.today().strftime('%Y-%m-%d'))

current_quarter_row = date_df[(date_df['first_date'] <= today) & (date_df['end_date'] >= today)]

if not current_quarter_row.empty:
    Q = current_quarter_row.iloc[0]['Q']
    first_date = current_quarter_row.iloc[0]['first_date']
    end_date = current_quarter_row.iloc[0]['end_date']
    print(f"Q: {Q}, First Date: {first_date}, End Date: {end_date}")
else:
    print("本日の日付に該当するクォーターは見つかりませんでした。")

#社員データ抽出
sql="""
select
user_id ,
sei_plus,
concat(sei,mei) as seimei
FROM `r-group-bigdata.live_company.syain`
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
syain_data = client.query(sql).result().to_dataframe()
syain_data.sample(30)

Q: 29-3Q, First Date: 2026-04-01, End Date: 2026-06-30


,user_id,sei_plus,seimei
4798,ry-kudo,工藤琉,工藤琉希
2953,y-ida,井田陽,井田陽介
2768,isozaki,磯崎ゆ,磯崎ゆう子
1816,a-nakajima,仲嶌亜,仲嶌亜由美
4298,h-matsuura,松浦広,松浦広聖
1269,hir-hashimoto,橋本裕,橋本裕美
84,hiromi-oode,大出弘,大出弘美
2447,hi-aoyama,青山ひ,青山光
709,sh-tanaka,田中慎,田中慎一
1543,kasuya,None,粕谷雄三


## 初期交渉データ

In [7]:
#初期交渉のDBから基本データを抽出（初期交渉データ）
#AP獲得者がいる場合は、AP担当はAP獲得者。
#AP獲得者が空欄でAPソース：パートナー紹介の場合はAP担当は紹介受領者。
#AP獲得者が空欄でAPソース：人事部、転機は面談担当者。
#AP獲得者が空欄でAPソースがパートナー紹介、人事部、転機の場合もAP担当は面談担当者。
#且つAP獲得者がロンザン所属でない場合（所属フラグ空欄）は面談担当者にする。

sql = """
WITH
-- CTE 1: APソースのマスタデータを準備
ap_source_master AS (
  SELECT
    code,
    name
  FROM `r-group-bigdata.live_rhs.sys_consts`
  WHERE group_code = 19),

-- CTE 2: 役職レイヤーのマスタデータを準備
layer_master AS (
  SELECT
    code,
    name
  FROM `r-group-bigdata.live_rhs.sys_consts`
  WHERE group_code = 25),

-- ※CTE 3 (first_interview_person) は不要になったため削除しました

-- CTE 3: メインとなる交渉データに必要な情報を付与し、基本的な変換処理を行う
base_data AS (
  SELECT
    shoki.id,
    shoki.tenki_id,
    shoki.kohosha_id,
    CASE
      WHEN consts.name = '転機社長名鑑' THEN '転機'
      WHEN consts.name = '人事部経由' THEN '人事部'
      WHEN consts.name IN ('顧問名鑑登録　解放者', '社外取締役名鑑　候補者') THEN '解放顧問'
      WHEN consts.name IN ('HP反響', '上場企業役員DM', 'Gアポ') THEN 'その他'
      ELSE consts.name END AS APsource,
    layer.name AS layer,
    shoki.annual_income,
    COALESCE(syi1.sei_plus, shoki.mendan_tanto) AS mendan_tanto,
    syi2.sei_plus AS ap_kakutokusha,
    shoki.kosho_setteibi,
    shoki.kosho_yoteibi,
    shoki.kosho_jisshibi,
    shoki.tsr_code,
    shoki.kosho_seq,
    shoki.saikosho_kaisu,
    shoki.saikosho_seq,
    shoki.valid_flag,
    -- ウィンドウ関数を使い、候補者ごとに前回交渉実施日からの経過日数を計算
    DATE_DIFF(
      shoki.kosho_setteibi,
      LAG(shoki.kosho_jisshibi) OVER (PARTITION BY shoki.kohosha_id ORDER BY shoki.kosho_setteibi),
      DAY) AS keikabi
  FROM `r-group-bigdata.live_rhs.shokikoshos` AS shoki
  LEFT JOIN `r-group-bigdata.live_rhs.kohoshas` AS khs ON shoki.kohosha_id = khs.id
  LEFT JOIN `r-group-bigdata.live_company.syain` AS syi1 ON shoki.mendan_tanto = syi1.user_id
  LEFT JOIN `r-group-bigdata.live_company.syain` AS syi2 ON shoki.ap_kakutoku = syi2.user_id
  -- first_interview_person の JOIN は削除しました
  LEFT JOIN ap_source_master AS consts ON shoki.ap_source = consts.code
  LEFT JOIN layer_master AS layer ON shoki.max_bushoyakushoku = layer.code),

-- CTE 4: 1つ前のAPsourceの値を取得
data_with_prev_apsource AS (
  SELECT
    *,
    -- ウィンドウ関数を使い、1つ前のAPsourceを取得
    LAG(APsource) OVER (PARTITION BY kohosha_id ORDER BY kosho_setteibi) AS prev_APsource
  FROM base_data),

-- CTE 5: APsourceが変更されたかどうかのフラグを計算
data_with_aps_change AS (
  SELECT
    *,
    -- APsourceが前回から変更された場合に1を立てる
    CASE
      WHEN APsource != prev_APsource THEN 1
      ELSE 0 END AS APS_change
  FROM data_with_prev_apsource),

-- 新設 CTE 6: first_mendan_tantoを計算する前に、まずここで sai_flg を確定させる
calc_sai_flg AS (
  SELECT
    *,
    CASE
      WHEN kosho_seq = 1 THEN 1 -- 初回交渉
      WHEN APS_change = 1 THEN 1 -- APsourceが変更された場合
      WHEN keikabi > 90 THEN 2 -- 前回実施から90日以上経過
      ELSE 0
    END AS sai_flg
  FROM data_with_aps_change)

-- 最終的なSELECT文
SELECT
  id,
  tenki_id,
  kohosha_id,
  APsource,
  layer,
  annual_income,
  mendan_tanto as kohosha_tanto,
  ap_kakutokusha,
  kosho_setteibi,
  kosho_yoteibi,
  kosho_jisshibi,
  kosho_seq,
  APS_change,
  keikabi,
  sai_flg,
  
  -- ★新しいロジック★
  -- 直近の `sai_flg = 1` になった時の `mendan_tanto` を取得して引き継ぐ
  LAST_VALUE(CASE WHEN sai_flg = 1 THEN mendan_tanto END IGNORE NULLS) OVER (
    PARTITION BY kohosha_id 
    ORDER BY kosho_setteibi 
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS first_mendan_tanto,
  
  tsr_code
FROM calc_sai_flg
# where kohosha_id = 38423
ORDER BY kosho_setteibi ASC;
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
shokikosho_data = client.query(sql).result().to_dataframe()


In [8]:
# ==========================================
# セル1: データ型の変換（より高速な一括処理）
# ==========================================
# 1. 'dbdate' 型のカラムを自動で探して、applyを使って一括で日付型に変換します
dbdate_columns = shokikosho_data.select_dtypes(include=['dbdate']).columns
shokikosho_data[dbdate_columns] = shokikosho_data[dbdate_columns].apply(pd.to_datetime)

# 2. 日付型「以外」のカラムをすべて 'object' 型に変換します
columns_to_convert = shokikosho_data.select_dtypes(exclude=['datetime', 'datetime64']).columns
shokikosho_data[columns_to_convert] = shokikosho_data[columns_to_convert].astype('object')

# 3. 確認
print(shokikosho_data.dtypes)

id                            object
tenki_id                      object
kohosha_id                    object
APsource                      object
layer                         object
annual_income                 object
kohosha_tanto                 object
ap_kakutokusha                object
kosho_setteibi        datetime64[ns]
kosho_yoteibi         datetime64[ns]
kosho_jisshibi        datetime64[ns]
kosho_seq                     object
APS_change                    object
keikabi                       object
sai_flg                       object
first_mendan_tanto            object
tsr_code                      object
dtype: object


In [9]:
# ==========================================
# セル2: Q_masterのマージ（メソッドチェーンでスッキリと）
# ==========================================
q_master_subset = Q_master[['日付', 'Q', '月', '営業日', '同営業日比較']]

# 設定日のマージとリネームを1つの処理で繋げて行います
shokikosho_data = (
    shokikosho_data.merge(q_master_subset, left_on='kosho_setteibi', right_on='日付', how='left')
    .drop(columns=['日付'])
    .rename(columns={'Q': 'setteibi_Q', '月': 'setteibi_月', '営業日': 'setteibi_営業日', '同営業日比較': 'setteibi_同営業日比較'})
)

# 実施日のマージとリネームも同様に行います
shokikosho_data = (
    shokikosho_data.merge(q_master_subset, left_on='kosho_jisshibi', right_on='日付', how='left')
    .drop(columns=['日付'])
    .rename(columns={'Q': 'jisshibi_Q', '月': 'jisshibi_月', '営業日': 'jisshibi_営業日', '同営業日比較': 'jisshibi_同営業日比較'})
)

In [10]:
# ==========================================
# セル3: master2（社員マスタ）のマージ
# ==========================================
master2_subset = master2[['Q', 'sei_plus', '職種', 'ロンザン所属フラグ', 'レイヤー']]

# 面談担当者情報のマージ
shokikosho_data = (
    shokikosho_data.merge(master2_subset, left_on=['setteibi_Q', 'kohosha_tanto'], right_on=['Q', 'sei_plus'], how='left')
    .drop(columns=['Q', 'sei_plus'])
    .rename(columns={'職種': '候担_職種', 'ロンザン所属フラグ': '候担_ロンザン所属フラグ', 'レイヤー': '候担_レイヤー'})
)

In [11]:
# ==========================================
# セル4: データの抽出、整形、縦積み（劇的にコードを削減）
# ==========================================
# 1. 対象データ(sai_flg == 1)を一度だけ抽出します
base_shoki = shokikosho_data[shokikosho_data['sai_flg'] == 1].copy()

# 2. 共通で引き継ぐカラム名と、新しく付与するキレイなカラム名のリストを準備します
common_cols = ['id', 'kohosha_id', 'APsource', 'kohosha_tanto', '候担_職種', '候担_ロンザン所属フラグ', '候担_レイヤー']
clean_cols = ['Q', '月', '営業日', '同営業日', '日付', 'KPI_ID', '候補者ID', 'APソース', '担当', '職種', '候担_ロンザン所属フラグ', 'レイヤー']

# 3. 設定データを作成（抽出と一括リネーム）
df_settei = base_shoki[['setteibi_Q', 'setteibi_月', 'setteibi_営業日', 'setteibi_同営業日比較', 'kosho_setteibi'] + common_cols].copy()
df_settei.columns = clean_cols  # カラム名を一気に上書き！
df_settei['type'] = 'shoki_settei'

# 4. 実施データを作成（抽出と一括リネーム）
df_jisshi = base_shoki[['jisshibi_Q', 'jisshibi_月', 'jisshibi_営業日', 'jisshibi_同営業日比較', 'kosho_jisshibi'] + common_cols].copy()
df_jisshi.columns = clean_cols  # カラム名を一気に上書き！
df_jisshi['type'] = 'shoki_jisshi'

# 5. データを縦に繋げ、共通の値(value)を入れます
shoki_combined_data = pd.concat([df_settei, df_jisshi], ignore_index=True)
shoki_combined_data['value'] = 1

# 結合後のデータを確認
print("結合後のデータの行数と列数:", shoki_combined_data.shape)
display(shoki_combined_data.head())


結合後のデータの行数と列数: (152952, 14)


,Q,月,営業日,同営業日,日付,KPI_ID,候補者ID,APソース,担当,職種,候担_ロンザン所属フラグ,レイヤー,type,value
0,NaN,NaT,NaN,NaN,NaT,90208,74564,SMAP,伊藤大２,NaN,NaN,NaN,shoki_settei,1
1,18-1Q,2014-11-01,26,,2014-11-20,637,1742,解放顧問,隆郁,NaN,NaN,NaN,shoki_settei,1
2,18-1Q,2014-12-01,32,,2014-12-01,2237,3613,パートナー紹介,角田隆,NaN,NaN,NaN,shoki_settei,1
3,18-2Q,2015-01-01,19,同営業日,2015-01-30,1135,2418,SMAP,隆郁,NaN,NaN,NaN,shoki_settei,1
4,18-2Q,2015-02-01,30,,2015-02-17,1975,3260,SMAP,隆郁,NaN,NaN,NaN,shoki_settei,1


In [12]:
# ==========================================
# 1. あなたが並べたい理想の順番を「リスト」で定義します
# 今後KPIが増えたら、ここにどんどんカンマ区切りで追記していくだけでOKです！
# ==========================================
type_order = [
    'shoki_settei',
    'shoki_jisshi'
]

# ==========================================
# 2. shoki_combined_data の 'type' カラムに、上で作った「独自の順番（ルール）」を記憶させます
# ==========================================
shoki_combined_data['type'] = pd.Categorical(
    shoki_combined_data['type'], 
    categories=type_order, 
    ordered=True  # 「この順番に意味があるよ（順序付きだよ）」と教えてあげます
)

In [13]:
# ==========================================
# セル5: ピボットテーブルの作成
# ==========================================
# df1: APソース
df1 = shoki_combined_data.pivot_table(
    index=["type", "APソース"],
    columns="Q",
    aggfunc="count",
    values="value"
).fillna(0).reset_index()

df1_fixed = df1.rename(columns={'APソース': '項目名'})
df1_fixed.insert(1, '集計軸', 'APソース') # 2列目に目印を追加

# df2: 職種
df2 = shoki_combined_data.pivot_table(
    index=["type", "職種"],
    columns="Q",
    aggfunc="count",
    values="value"
).fillna(0).reset_index()

df2_fixed = df2.rename(columns={'職種': '項目名'})
df2_fixed.insert(1, '集計軸', '職種')

# df3: レイヤー
df3 = shoki_combined_data.pivot_table(
    index=["type", "レイヤー"],
    columns="Q",
    aggfunc="count",
    values="value"
).fillna(0).reset_index()

df3_fixed = df3.rename(columns={'レイヤー': '項目名'})
df3_fixed.insert(1, '集計軸', 'レイヤー')

# df4: 担当
df4 = shoki_combined_data.pivot_table(
    index=["type", "担当"],
    columns="Q",
    aggfunc="count",
    values="value"
).fillna(0).reset_index()

# 1. master2から「sei_plus」の重複のないリスト（名簿）を作成します
valid_members = master2['sei_plus'].dropna().unique()

# 2. df4の「担当」が、上で作った名簿に含まれている（isin）行だけを残します
df4 = df4[df4['担当'].isin(valid_members)].copy()

df4_fixed = df4.rename(columns={'担当': '項目名'})
df4_fixed.insert(1, '集計軸', '担当')

# ==========================================
# 縦にガッチャンコ（結合）
# ==========================================
all_pivot_df = pd.concat([df1_fixed, df2_fixed, df3_fixed, df4_fixed], ignore_index=True)

# 1. 'type' 列を並び替え用のカテゴリ型に変換（空欄があってもエラーにならないようにします）
all_pivot_df['type'] = pd.Categorical(all_pivot_df['type'], categories=type_order, ordered=True)

# 2. 数値が入るべき列（Qの列）だけを特定して、空欄を 0 で埋めます
# type, 集計軸, 項目名 以外の列が対象になります
numeric_cols = all_pivot_df.columns.difference(['type', '集計軸', '項目名'])
all_pivot_df[numeric_cols] = all_pivot_df[numeric_cols].fillna(0)

# 3. 独自の順番通りに並び替えを実行
all_pivot_df = all_pivot_df.sort_values(by=['type', '集計軸', '項目名'])

# 4. 見た目を整える
all_pivot_df.columns.name = None

# 初期交渉の結果を別名で保存しておきます
shoki_pivot_df = all_pivot_df.copy()

# 確認表示
display(shoki_pivot_df.head())

C:\Users\suehara\AppData\Local\Temp\ipykernel_19584\1568351815.py:5: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  df1 = shoki_combined_data.pivot_table(
C:\Users\suehara\AppData\Local\Temp\ipykernel_19584\1568351815.py:16: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  df2 = shoki_combined_data.pivot_table(
C:\Users\suehara\AppData\Local\Temp\ipykernel_19584\1568351815.py:27: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  df3 = shoki_combined_data.pivot_table(
C:\Users\suehara\AppData\Local\Temp\ipykerne

,type,集計軸,項目名,18-1Q,18-2Q,18-3Q,18-4Q,19-1Q,19-2Q,19-3Q,19-4Q,20-1Q,20-2Q,20-3Q,20-4Q,21-1Q,21-2Q,21-3Q,21-4Q,22-1Q,22-2Q,22-3Q,22-4Q,23-1Q,23-2Q,23-3Q,23-4Q,24-1Q,24-2Q,24-3Q,24-4Q,25-1Q,25-2Q,25-3Q,25-4Q,26-1Q,26-2Q,26-3Q,26-4Q,27-1Q,27-2Q,27-3Q,27-4Q,28-1Q,28-2Q,28-3Q,28-4Q,29-1Q,29-2Q,29-3Q
0,shoki_settei,APソース,SMAP,0,2,0,0,0,10,43,17,36,58,40,41,36,50,42,76,71,97,112,101,106,96,180,144,231,265,185,334,183,184,203,508,876,1173,928,930,1128,1192,1282,1414,1429,1427,1545,1568,1622,1577,621
1,shoki_settei,APソース,その他,0,0,0,1,1,11,26,30,57,118,68,142,77,132,115,106,76,64,61,72,65,72,83,72,96,114,87,59,64,64,48,66,54,27,45,44,62,85,82,92,79,95,105,132,83,91,24
2,shoki_settei,APソース,パートナー紹介,1,1,0,0,11,68,90,36,70,171,151,247,252,202,368,294,324,269,164,201,236,190,249,183,315,426,377,340,203,173,221,225,172,175,122,156,149,254,276,230,149,143,119,140,147,137,52
3,shoki_settei,APソース,人事部,0,0,0,0,2,17,13,22,25,39,33,34,30,57,41,72,57,89,53,91,97,78,81,86,125,126,100,79,99,88,62,77,102,109,69,57,82,73,56,81,85,70,51,61,108,162,21
4,shoki_settei,APソース,解放顧問,1,0,0,0,0,2,21,3,18,24,36,50,82,117,87,66,37,24,32,25,61,33,90,20,152,89,60,79,65,9,48,79,105,40,98,84,54,107,55,43,24,27,28,37,31,10,3


## 本交渉データ

In [14]:
sql = """
WITH
-- =================================================================
-- マスタデータ準備セクション
-- 必要なマスタデータを事前にCTEとして定義し、再利用しやすくする
-- =================================================================

-- CTE 1: APソースのマスタデータ (group_code = 19)
ap_source_master AS (
  SELECT code, name FROM `r-group-bigdata.live_rhs.sys_consts` WHERE group_code = 19
),

-- CTE 2: 役職レイヤーのマスタデータ (group_code = 25)
layer_master AS (
  SELECT code, name FROM `r-group-bigdata.live_rhs.sys_consts` WHERE group_code = 25
),

-- CTE 3: ヨミのマスタデータ (group_code = 9)
yomi_master AS (
  SELECT code, name FROM `r-group-bigdata.live_rhs.sys_consts` WHERE group_code = 9
),

-- CTE 4: 役職クラスのマスタデータ (group_code = 10)
yakushoku_class_master AS (
  SELECT code, name FROM `r-group-bigdata.live_rhs.sys_consts` WHERE group_code = 10
),

-- =================================================================
-- 初期交渉データ準備セクション (元の`SHOKIS` CTEに相当)
-- =================================================================

-- CTE 5: 候補者ごとの初回接触日を計算
shokikoshos_with_initial_date AS (
  SELECT
    *,
    FIRST_VALUE(kosho_jisshibi IGNORE NULLS) OVER (PARTITION BY kohosha_id ORDER BY kosho_jisshibi) AS initial_contact_date
  FROM
    `r-group-bigdata.live_rhs.shokikoshos`
),

-- CTE 6: 初期交渉データの中間処理 (フラグ計算の前段階)
shoki_base AS (
  SELECT
    shk.id,
    shk.tenki_id,
    shk.kohosha_id,
    shk.ap_source,
    layer.name AS layer,
    shk.annual_income,
    COALESCE(syi1.sei_plus, shk.mendan_tanto) AS mendan_tanto,
    syi2.sei_plus AS ap_kakutokusha,
    shk.kosho_setteibi,
    shk.kosho_yoteibi,
    shk.kosho_jisshibi,
    shk.kosho_seq,
    shk.saikosho_kaisu,
    shk.saikosho_seq,
    shk.valid_flag,
    shk.initial_contact_date,
    -- 前回交渉実施日からの経過日数を計算
    DATE_DIFF(shk.kosho_setteibi, LAG(shk.kosho_jisshibi) OVER (PARTITION BY shk.kohosha_id ORDER BY shk.kosho_jisshibi), DAY) AS keikabi
  FROM
    shokikoshos_with_initial_date AS shk
    LEFT JOIN `r-group-bigdata.live_company.syain` AS syi1 ON shk.mendan_tanto = syi1.user_id
    LEFT JOIN `r-group-bigdata.live_company.syain` AS syi2 ON shk.ap_kakutoku = syi2.user_id
    LEFT JOIN layer_master AS layer ON shk.max_bushoyakushoku = layer.code
),

-- CTE 7: 初期交渉データの sai_flg を計算
shoki_calc_sai_flg AS (
  SELECT
    *,
    -- sai_flgを計算
    CASE
      WHEN kosho_seq = 1 THEN 1 -- 初回交渉
      -- APソースが前回から変更された場合
      WHEN ap_source != LAG(ap_source) OVER (PARTITION BY kohosha_id ORDER BY kosho_setteibi) THEN 1
      WHEN keikabi > 90 THEN 2 -- 前回実施から90日以上経過
      ELSE 0
    END AS sai_flg
  FROM
    shoki_base
),

-- 新設 CTE 7.5: 初期交渉データ内で first_mendan_tanto を引き継ぎ計算する
shoki_final AS (
  SELECT
    *,
    -- 直近の `sai_flg = 1` になった時の `mendan_tanto` を取得して引き継ぐ
    LAST_VALUE(CASE WHEN sai_flg = 1 THEN mendan_tanto END IGNORE NULLS) OVER (
      PARTITION BY kohosha_id 
      ORDER BY kosho_setteibi 
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS first_mendan_tanto
  FROM
    shoki_calc_sai_flg
),

-- =================================================================
-- 本交渉データ準備セクション (元の`HONS` CTEに相当)
-- =================================================================

-- CTE 8: 案件ごとの最新のヨミを取得
latest_yomis AS (
  SELECT
    anken_id,
    yomi
  FROM (
    SELECT
      anken_id,
      yomi,
      ROW_NUMBER() OVER (PARTITION BY anken_id ORDER BY yomi_torokubi DESC) AS rn
    FROM
      `r-group-bigdata.live_rhs.honkosho_yomis`
  )
  WHERE
    rn = 1
),

-- CTE 10: 本交渉データと関連データを結合
hons_base AS (
  SELECT
    hon.id AS honkosho_id,
    hon.anken_id,
    an.kohosha_id,
    an.linked_shokikosho_id,
    kgy.tsr_code,
    kgy.name AS kigyo_name,
    -- APソースを初期交渉と候補者マスタから取得し、優先度付け
    COALESCE(consts1.name, consts2.name) AS AP_source_raw,
    shoki.annual_income,
    shoki.layer,
    shoki.initial_contact_date,
    shoki.kosho_setteibi AS shoki_setteibi,
    shoki.kosho_jisshibi AS shoki_jisshibi,
    hon.kosho_setteibi AS hon_setteibi,
    hon.kosho_yoteibi AS hon_yoteibi,
    hon.kosho_jisshibi AS hon_jisshibi,
    COALESCE(syi1.sei_plus, hon.kohosha_tanto) AS kohosha_tanto,
    syi2.sei_plus AS kigyo_tanto,
    hon.kosho_seq AS hon_seq,
    shoki.kosho_seq AS shoki_seq,
    shoki.sai_flg,
    DATE_DIFF(hon.kosho_setteibi, shoki.kosho_jisshibi, DAY) AS jisshibi_sa,
    yomi1.name AS yomi,
    -- 最終的なヨミを決定
    COALESCE(yomi2.name, yomi1.name) AS yomi_final,
    ROUND(tsr.tokikessan_uriagedaka / 100000, 0) AS uriage,
    consts3.name AS yakushoku_class,
    -- shoki_final で計算した first_mendan_tanto を取得
    shoki.first_mendan_tanto,
        case when an.hanjokin = 20 then "半常勤"
         else "" end as hanjokin
  FROM
    `r-group-bigdata.live_rhs.honkoshos` AS hon
    LEFT JOIN `r-group-bigdata.live_rhs.ankens` AS an ON hon.anken_id = an.id
    LEFT JOIN shoki_final AS shoki ON an.linked_shokikosho_id = shoki.id
    LEFT JOIN `r-group-bigdata.live_rhs.kigyos` AS kgy ON an.kigyo_id = kgy.id
    LEFT JOIN `r-group-bigdata.live_rhs.kohoshas` AS koho ON an.kohosha_id = koho.id
    LEFT JOIN `r-group-bigdata.tsr.company_info` AS tsr ON kgy.tsr_code = tsr.tsr_code
    LEFT JOIN `r-group-bigdata.live_company.syain` AS syi1 ON hon.kohosha_tanto = syi1.user_id
    LEFT JOIN `r-group-bigdata.live_company.syain` AS syi2 ON an.kigyo_tanto = syi2.user_id
    LEFT JOIN latest_yomis AS ly ON an.id = ly.anken_id
    -- マスタ結合
    LEFT JOIN ap_source_master AS consts1 ON shoki.ap_source = consts1.code
    LEFT JOIN ap_source_master AS consts2 ON koho.ap_source = consts2.code
    LEFT JOIN yomi_master AS yomi1 ON hon.yomi = yomi1.code
    LEFT JOIN yomi_master AS yomi2 ON ly.yomi = yomi2.code
    LEFT JOIN yakushoku_class_master AS consts3 ON hon.yakushoku_class = consts3.code
  WHERE hon.kosho_seq = 1
)

-- =================================================================
-- 最終的な出力
-- =================================================================
SELECT
  honkosho_id as id,
  anken_id,
  kohosha_id,
  linked_shokikosho_id,
  tsr_code,
  kigyo_name,
  annual_income,
  layer,
  initial_contact_date,
  
  -- ★修正ポイント★ FORMAT_DATE（文字化）を外し、CASTで純粋な日付型（DATE）として出力します
  CAST(shoki_setteibi AS DATE) AS shoki_setteibi,
  CAST(shoki_jisshibi AS DATE) AS shoki_jisshibi,
  CAST(hon_setteibi AS DATE) AS hon_setteibi,
  CAST(hon_yoteibi AS DATE) AS hon_yoteibi,
  CAST(hon_jisshibi AS DATE) AS hon_jisshibi,
  
  kohosha_tanto,
  kigyo_tanto,
  hon_seq,
  shoki_seq,
  jisshibi_sa,
  yomi,
  yomi_final,
  uriage,
  yakushoku_class,
  first_mendan_tanto,
  hanjokin,
  -- APsourceを分かりやすいカテゴリに分類
  CASE
    WHEN AP_source_raw = '転機社長名鑑' THEN '転機'
    WHEN AP_source_raw = '人事部経由' THEN '人事部紹介'
    WHEN AP_source_raw IN ('顧問名鑑登録　解放者', '社外取締役名鑑　候補者') THEN '顧問名鑑登録者'
    WHEN AP_source_raw IN ('HP反響', '上場企業役員DM', 'Gアポ') THEN 'その他'
    ELSE AP_source_raw
  END AS APsource,
  -- 最終的なフラグ `sai_flg2` を計算
  CASE
    WHEN shoki_jisshibi IS NULL THEN 2
    WHEN jisshibi_sa > 90 THEN 2
    ELSE sai_flg
  END AS sai_flg2
FROM hons_base
ORDER BY
  anken_id;
"""

client = bigquery.Client(credentials=credentials, project=credentials.project_id)
honkosho_data = client.query(sql).result().to_dataframe()

In [15]:
honkosho_data.dtypes

id                        Int64
anken_id                  Int64
kohosha_id                Int64
linked_shokikosho_id      Int64
tsr_code                 object
kigyo_name               object
annual_income           float64
layer                    object
initial_contact_date     dbdate
shoki_setteibi           dbdate
shoki_jisshibi           dbdate
hon_setteibi             dbdate
hon_yoteibi              dbdate
hon_jisshibi             dbdate
kohosha_tanto            object
kigyo_tanto              object
hon_seq                   Int64
shoki_seq                 Int64
jisshibi_sa               Int64
yomi                     object
yomi_final               object
uriage                  float64
yakushoku_class          object
first_mendan_tanto       object
hanjokin                 object
APsource                 object
sai_flg2                  Int64
dtype: object

In [16]:
# ==========================================
# セル1: データ型の変換（より高速な一括処理）
# ==========================================
# 1. 'dbdate' 型のカラムを自動で探して、applyを使って一括で日付型に変換します
dbdate_columns = honkosho_data.select_dtypes(include=['dbdate']).columns
honkosho_data[dbdate_columns] = honkosho_data[dbdate_columns].apply(pd.to_datetime)

# 2. 日付型「以外」のカラムをすべて 'object' 型に変換します
columns_to_convert = honkosho_data.select_dtypes(exclude=['datetime', 'datetime64']).columns
honkosho_data[columns_to_convert] = honkosho_data[columns_to_convert].astype('object')

# 3. 確認
print(honkosho_data.dtypes)

id                              object
anken_id                        object
kohosha_id                      object
linked_shokikosho_id            object
tsr_code                        object
kigyo_name                      object
annual_income                   object
layer                           object
initial_contact_date    datetime64[ns]
shoki_setteibi          datetime64[ns]
shoki_jisshibi          datetime64[ns]
hon_setteibi            datetime64[ns]
hon_yoteibi             datetime64[ns]
hon_jisshibi            datetime64[ns]
kohosha_tanto                   object
kigyo_tanto                     object
hon_seq                         object
shoki_seq                       object
jisshibi_sa                     object
yomi                            object
yomi_final                      object
uriage                          object
yakushoku_class                 object
first_mendan_tanto              object
hanjokin                        object
APsource                 

In [ ]:
# ==========================================
# セル2: Q_masterのマージ（メソッドチェーンでスッキリと）
# ==========================================

# 設定日のマージとリネームを1つの処理で繋げて行います
honkosho_data = (
    honkosho_data.merge(q_master_subset, left_on='hon_setteibi', right_on='日付', how='left')
    .drop(columns=['日付'])
    .rename(columns={'Q': 'setteibi_Q', '月': 'setteibi_月', '営業日': 'setteibi_営業日', '同営業日比較': 'setteibi_同営業日比較'})
)

# 実施日のマージとリネームも同様に行います
honkosho_data = (
    honkosho_data.merge(q_master_subset, left_on='hon_jisshibi', right_on='日付', how='left')
    .drop(columns=['日付'])
    .rename(columns={'Q': 'jisshibi_Q', '月': 'jisshibi_月', '営業日': 'jisshibi_営業日', '同営業日比較': 'jisshibi_同営業日比較'})
)

In [18]:
# ==========================================
# セル3: master2（社員マスタ）のマージ
# ==========================================

# 候補者担当者情報のマージ
honkosho_data = (
    honkosho_data.merge(master2_subset, left_on=['setteibi_Q', 'kohosha_tanto'], right_on=['Q', 'sei_plus'], how='left')
    .drop(columns=['Q', 'sei_plus'])
    .rename(columns={'職種': '候担_職種', 'ロンザン所属フラグ': '候担_ロンザン所属フラグ', 'レイヤー': '候担_レイヤー'})
)

# 企業担当者情報のマージ
honkosho_data = (
    honkosho_data.merge(master2_subset, left_on=['setteibi_Q', 'kigyo_tanto'], right_on=['Q', 'sei_plus'], how='left')
    .drop(columns=['Q', 'sei_plus'])
    .rename(columns={'職種': '企担_職種', 'ロンザン所属フラグ': '企担_ロンザン所属フラグ', 'レイヤー': '企担_レイヤー'})
)



In [19]:
# ==========================================
# 新しいカラム「組み手」を追加する処理
# ==========================================

# 1. 組み手を判定するための独自のルール（関数）を作ります
def judge_kumite(row):
    # 安全に判定するため、値を一度「文字」として取り出します
    koho_flag = str(row['候担_ロンザン所属フラグ'])
    kigyo_flag = str(row['企担_ロンザン所属フラグ'])
    
    # ルール1: 両方とも '1' の場合（"1.0" となっているケースも考慮して "1" が含まれるかチェックします）
    if '1' in koho_flag and '1' in kigyo_flag:
        return '両手'
    
    # ルール2: どちらか片方でも '1' の場合（上の条件を抜けたものがここに来ます）
    elif '1' in koho_flag or '1' in kigyo_flag:
        return '片手'
    
    # ルール3: それ以外（両方 '0' や、空欄など）の場合
    else:
        return '無効'

# 2. 作ったルール（judge_kumite）を、データフレームの行ごと（axis=1）に適用（apply）します
honkosho_data['組み手'] = honkosho_data.apply(judge_kumite, axis=1)


In [20]:
display(honkosho_data.sample(10))

,id,anken_id,kohosha_id,linked_shokikosho_id,tsr_code,kigyo_name,annual_income,layer,initial_contact_date,shoki_setteibi,shoki_jisshibi,hon_setteibi,hon_yoteibi,hon_jisshibi,kohosha_tanto,kigyo_tanto,hon_seq,shoki_seq,jisshibi_sa,yomi,yomi_final,uriage,yakushoku_class,first_mendan_tanto,hanjokin,APsource,sai_flg2,setteibi_Q,setteibi_月,setteibi_営業日,setteibi_同営業日比較,jisshibi_Q,jisshibi_月,jisshibi_営業日,jisshibi_同営業日比較,候担_職種,候担_ロンザン所属フラグ,候担_レイヤー,企担_職種,企担_ロンザン所属フラグ,企担_レイヤー,組み手
20216,21282,20326,56589,67400,292098618,（株）マルエイ商事,1200.0,本体部長クラス,2024-08-07,2024-08-02,2024-08-07,2024-08-19,2024-08-22,2024-08-22,榎本直,加藤太,1,1,12,D+,D+,27.0,会長・社長,榎本直,,SMAP,1,27-4Q,2024-08-01,30,,27-4Q,2024-08-01,33,,ミドル候B,1,023生,NaN,NaN,NaN,片手
21059,22164,21174,57319,68392,870329243,新日本製薬（株）,1280.0,課長クラス,2024-09-03,2024-08-30,2024-09-03,2024-11-12,2024-11-22,2024-11-22,高橋優２,黒部智,1,1,70,D+,D+,403.0,役員（決裁権無し）,高橋優２,,転機,1,28-1Q,2024-11-01,27,,28-1Q,2024-11-01,35,,ミドル企業,1,中途：27期前期,ミドル企業,1,中途：27期前期,両手
2475,2576,2507,5626,3987,291014500,八洲運輸（株）,1300.0,子会社社長・役員クラス,2018-01-26,2018-01-19,2018-01-26,2018-03-12,2018-03-14,2018-03-14,松尾幸,後田孝,1,1,45,D,B,24.0,会長・社長,松尾幸,,転機,1,21-2Q,2018-03-01,45,,21-2Q,2018-03-01,47,,,1,None,NaN,NaN,NaN,片手
21719,22858,21836,56152,66820,870356860,（株）千鳥饅頭総本舗,950.0,本体部長クラス,2024-07-23,2024-07-19,2024-07-23,2025-01-20,2025-01-22,2025-01-22,五十嵐り,岸靖,1,1,181,オチ（企）,オチ（企）,26.0,会長・社長,五十嵐り,,SMAP,2,28-2Q,2025-01-01,11,同営業日,28-2Q,2025-01-01,13,同営業日,ミドル候B,1,中途：27期前期,フロント,1,部責,両手
6522,7043,6577,15774,14218,610138090,菱岡工業（株）,1300.0,子会社部長クラス,2019-11-13,2019-11-13,2019-11-13,2019-11-22,2019-12-06,2019-12-06,中川美,福田達,1,1,9,オチ（企）,オチ（企）,64.0,None,中川美,,人事部紹介,1,23-1Q,2019-11-01,33,,23-1Q,2019-12-01,43,,ミドル候B,1,019生,NaN,NaN,NaN,片手
19866,20920,19972,52395,66665,342010034,（株）ヴェルシーナ,1246.0,本体部長クラス,2024-03-26,2024-07-16,2024-07-16,2024-07-16,2024-07-17,2024-07-17,小林翔,笹原啓,1,2,0,D+,成約,12.0,会長・社長,徳貞清,,SMAP,2,27-4Q,2024-07-01,11,同営業日,27-4Q,2024-07-01,12,同営業日,ミドル候B,1,023生,フロント,1,中途：22期後期,両手
8646,9294,8708,13008,11252,570180678,富士電子工業（株）,1100.0,それ以外,2019-06-18,2019-06-11,2019-06-18,2020-08-21,2020-08-29,2020-08-29,高見澤裕,磯崎ゆ,1,1,430,D+,D+,28.0,会長・社長,高見澤裕,,転機,2,23-4Q,2020-08-01,34,,23-4Q,2020-08-01,39,,ミドル候B,1,中途,フロント,1,既存,両手
8000,8602,8061,19525,-1,320043525,フジフーズ（株）,NaN,None,NaT,NaT,NaT,2020-05-29,2020-06-09,NaT,増田智,鳥羽亮,1,<NA>,<NA>,None,None,1238.0,役員（決裁権有り）,None,,SMAP,2,23-3Q,2020-05-01,38,,NaN,NaT,NaN,NaN,ミドル候B,1,中途,NaN,NaN,NaN,片手
18871,19895,18974,52915,62592,274826119,（株）ログ,980.0,それ以外,2024-04-10,2024-04-09,2024-04-10,2024-04-12,2024-04-20,2024-04-20,曽根裕,曽根裕,1,1,2,D+,成約,33.0,会長・社長,曽根裕,,パートナー紹介,1,27-3Q,2024-04-01,8,同営業日,27-3Q,2024-04-01,13,同営業日,ミドル企業,1,中途：26期後期,ミドル企業,1,中途：26期後期,両手
4565,4912,4607,532,470,340019336,（株）中村製作所,1000.0,本体役員クラス,2016-06-13,2017-02-09,2017-02-09,2019-03-20,2019-04-11,2019-04-11,長崎文,安田信,1,2,769,D+,D+,24.0,役員（決裁権有り）,長崎文,,その他,2,22-2Q,2019-03-01,51,,22-3Q,2019-04-01,6,同営業日,,1,None,,0,None,片手


In [21]:
# ==========================================
# セル4: データの抽出、整形、縦積み（関数化で劇的にコード削減！）
# ==========================================

# 1. 対象データを一度だけ抽出します
base_hon = honkosho_data.copy()

# 2. 共通で引き継ぐカラム名と、新しく付与するキレイなカラム名のリスト
common_cols = [
    'id', 'kohosha_id', 'tsr_code', 'anken_id', 'APsource', 'kohosha_tanto', 'kigyo_tanto', # ← 'tsr_code' を追加！
    '候担_職種', '候担_ロンザン所属フラグ', '候担_レイヤー', 
    '企担_職種', '企担_ロンザン所属フラグ', '企担_レイヤー', '組み手'
]
clean_cols = [
    'Q', '月', '営業日', '同営業日', '日付',           
    'KPI_ID', '候補者ID', 'tsr_code', '案件ID', 'APソース', '候補者担当', '企業担当', # ← ここにも 'tsr_code' を追加！
    '候担_職種', '候担_ロンザン所属フラグ', '候担_レイヤー',       
    '企担_職種', '企担_ロンザン所属フラグ', '企担_レイヤー', '組み手'   
]

# ==========================================
# ★ここが魔法の工場（関数）です！
# 条件(condition) と type名(type_name) を渡すと、自動で整形されたデータを作ります
# ==========================================
def make_df(condition, type_name, date_type='settei'):
    # 設定日ベースか、実施日ベースかで使う日付カラムを自動で切り替えます
    if date_type == 'settei':
        date_cols = ['setteibi_Q', 'setteibi_月', 'setteibi_営業日', 'setteibi_同営業日比較', 'hon_setteibi']
    else:
        date_cols = ['jisshibi_Q', 'jisshibi_月', 'jisshibi_営業日', 'jisshibi_同営業日比較', 'hon_jisshibi']
        
    # 条件で絞り込み、必要なカラムだけ抽出
    df_temp = base_hon[condition][date_cols + common_cols].copy()
    
    # リネームしてtypeをセット
    df_temp.columns = clean_cols
    df_temp['type'] = type_name
    
    return df_temp

# ==========================================
# 3. 必要なtypeのデータフレームをどんどん作って、リスト(dfs)に放り込みます
# ==========================================
dfs = []
mask_all = pd.Series(True, index=base_hon.index) # 「全件」を表すおまじない

# '半常勤' と一致するもの
mask_han = base_hon['hanjokin'] == '半常勤'
# それ以外（~ は「否定（Not）」を意味します。空白やNaNもこちらに含まれます）
mask_not_han = ~mask_han

# --- 基本 ---
dfs.append(make_df(mask_all & mask_not_han, 'hon_settei'))
dfs.append(make_df(mask_all & mask_not_han, 'hon_jisshi', date_type='jisshi')) # 実施データも残す

# --- 新規/再 ---
dfs.append(make_df((base_hon['sai_flg2'] != 2) & mask_not_han, 'hon_settei_shin'))
dfs.append(make_df((base_hon['sai_flg2'] == 2) & mask_not_han, 'hon_settei_sai'))

# --- 片手 ---
dfs.append(make_df((base_hon['組み手'] == '片手') & mask_not_han, 'hon_settei_kata'))
dfs.append(make_df((base_hon['組み手'] == '片手') & (base_hon['sai_flg2'] != 2) & mask_not_han, 'hon_settei_kata_shin'))
dfs.append(make_df((base_hon['組み手'] == '片手') & (base_hon['sai_flg2'] == 2) & mask_not_han, 'hon_settei_kata_sai'))

# --- 両手 ---
dfs.append(make_df((base_hon['組み手'] == '両手') & mask_not_han, 'hon_settei_ryo'))
dfs.append(make_df((base_hon['組み手'] == '両手') & (base_hon['sai_flg2'] != 2) & mask_not_han, 'hon_settei_ryo_shin'))
dfs.append(make_df((base_hon['組み手'] == '両手') & (base_hon['sai_flg2'] == 2) & mask_not_han, 'hon_settei_ryo_sai'))

# --- 追加分（社数・人数） ---
dfs.append(make_df(mask_all & mask_not_han, 'hon_settei_sha'))
dfs.append(make_df(mask_all & mask_not_han, 'hon_settei_nin'))

# --- 企業担当視点 ---
mask_kigyo = mask_all & (base_hon['企担_ロンザン所属フラグ'].astype(str).str.contains('1'))

dfs.append(make_df(mask_kigyo & mask_not_han, 'kigyo_settei'))
dfs.append(make_df(mask_kigyo & mask_not_han, 'kigyo_settei_sha'))
dfs.append(make_df(mask_kigyo & mask_not_han, 'kigyo_settei_nin'))

# ==========================================
# 3.1 新規追加：半常勤の項目の集計
# ==========================================
# --- 基本（半常勤） ---
dfs.append(make_df(mask_all & mask_han, 'hon_settei_han'))
# ※ もし hon_jisshi_han も必要であれば同様に追加可能です

# --- 新規/再（半常勤） ---
dfs.append(make_df((base_hon['sai_flg2'] != 2) & mask_han, 'hon_settei_han_shin'))
dfs.append(make_df((base_hon['sai_flg2'] == 2) & mask_han, 'hon_settei_han_sai'))

# --- 片手（半常勤） ---
dfs.append(make_df((base_hon['組み手'] == '片手') & mask_han, 'hon_settei_han_kata'))
dfs.append(make_df((base_hon['組み手'] == '片手') & (base_hon['sai_flg2'] != 2) & mask_han, 'hon_settei_han_kata_shin'))
dfs.append(make_df((base_hon['組み手'] == '片手') & (base_hon['sai_flg2'] == 2) & mask_han, 'hon_settei_han_kata_sai'))

# --- 両手（半常勤） ---
dfs.append(make_df((base_hon['組み手'] == '両手') & mask_han, 'hon_settei_han_ryo'))
dfs.append(make_df((base_hon['組み手'] == '両手') & (base_hon['sai_flg2'] != 2) & mask_han, 'hon_settei_han_ryo_shin'))
dfs.append(make_df((base_hon['組み手'] == '両手') & (base_hon['sai_flg2'] == 2) & mask_han, 'hon_settei_han_ryo_sai'))

# --- 追加分（社数・人数）（半常勤） ---
dfs.append(make_df(mask_all & mask_han, 'hon_settei_han_sha'))
dfs.append(make_df(mask_all & mask_han, 'hon_settei_han_nin'))

# --- 企業担当視点（半常勤） ---
dfs.append(make_df(mask_kigyo & mask_han, 'kigyo_settei_han'))
dfs.append(make_df(mask_kigyo & mask_han, 'kigyo_settei_han_sha'))
dfs.append(make_df(mask_kigyo & mask_han, 'kigyo_settei_han_nin'))

# 今後「hon_settei_kigyo」などが増える場合は、ここに dfs.append(...) を1行足すだけでOKです！

# ==========================================
# 4. まとめて縦にガッチャンコ
# ==========================================
hon_combined_data = pd.concat(dfs, ignore_index=True)
hon_combined_data['value'] = 1

# ==========================================
# ★集計マジック：指標によって「何を数えるか」を自動で切り替える仕込み
# ==========================================
# ① 全行に、絶対に重複しない「連番」を振る（通常の count と同じ挙動にさせるため）
hon_combined_data['calc_target'] = range(len(hon_combined_data))

# ② 「_sha」で終わる type は、数えるターゲットを「tsr_code」に上書き
mask_sha = hon_combined_data['type'].str.endswith('_sha', na=False)
hon_combined_data.loc[mask_sha, 'calc_target'] = hon_combined_data.loc[mask_sha, 'tsr_code']

# ③ 「_nin」で終わる type は、数えるターゲットを「候補者ID」に上書き
mask_nin = hon_combined_data['type'].str.endswith('_nin', na=False)
hon_combined_data.loc[mask_nin, 'calc_target'] = hon_combined_data.loc[mask_nin, '候補者ID']  # ← ここを日本語の '候補者ID' にしました！


# 結合後のデータを確認
print("結合後のデータの行数と列数:", hon_combined_data.shape)
display(hon_combined_data.head())

結合後のデータの行数と列数: (226485, 22)


C:\Users\suehara\AppData\Local\Temp\ipykernel_19584\2302710035.py:127: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['410147966' '292368810' '422112860' ... '296691348' '292905971'
 '571204368']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  hon_combined_data.loc[mask_sha, 'calc_target'] = hon_combined_data.loc[mask_sha, 'tsr_code']


,Q,月,営業日,同営業日,日付,KPI_ID,候補者ID,tsr_code,案件ID,APソース,候補者担当,企業担当,候担_職種,候担_ロンザン所属フラグ,候担_レイヤー,企担_職種,企担_ロンザン所属フラグ,企担_レイヤー,組み手,type,value,calc_target
0,19-4Q,2016-07-01,3,同営業日,2016-07-05,1,663,410147966,1,パートナー紹介,角田隆,北健,ミドル企業,1,None,NaN,NaN,NaN,片手,hon_settei,1,0
1,19-4Q,2016-07-01,3,同営業日,2016-07-05,2,165,292368810,2,パートナー紹介,角田隆,嵯峨優,ミドル企業,1,None,NaN,NaN,NaN,片手,hon_settei,1,1
2,19-4Q,2016-07-01,3,同営業日,2016-07-05,3,664,422112860,3,顧問名鑑登録者,大矢裕,岡大,ミドル企業,1,None,NaN,NaN,NaN,片手,hon_settei,1,2
3,19-4Q,2016-07-01,3,同営業日,2016-07-05,4,455,400652951,4,その他,大仲研,大仲研,ミドル企業,1,None,ミドル企業,1,None,両手,hon_settei,1,3
4,19-4Q,2016-07-01,3,同営業日,2016-07-05,5,665,422112860,5,パートナー紹介,角田隆,岡大,ミドル企業,1,None,NaN,NaN,NaN,片手,hon_settei,1,4


In [22]:
mask_sha

0         False
1         False
2         False
3         False
4         False
          ...  
226480    False
226481    False
226482    False
226483    False
226484    False
Name: type, Length: 226485, dtype: bool

In [23]:

hon_combined_data.loc[mask_sha, 'calc_target'] = hon_combined_data.loc[mask_sha, 'tsr_code']

# ③ 「_nin」で終わる type は、数えるターゲットを「kohosha_id」に上書き
mask_nin = hon_combined_data['type'].str.endswith('_nin', na=False)
hon_combined_data.loc[mask_nin, 'calc_target'] = hon_combined_data.loc[mask_nin, '候補者ID']


# 結合後のデータを確認
print("結合後のデータの行数と列数:", hon_combined_data.shape)
display(hon_combined_data.head())

結合後のデータの行数と列数: (226485, 22)


,Q,月,営業日,同営業日,日付,KPI_ID,候補者ID,tsr_code,案件ID,APソース,候補者担当,企業担当,候担_職種,候担_ロンザン所属フラグ,候担_レイヤー,企担_職種,企担_ロンザン所属フラグ,企担_レイヤー,組み手,type,value,calc_target
0,19-4Q,2016-07-01,3,同営業日,2016-07-05,1,663,410147966,1,パートナー紹介,角田隆,北健,ミドル企業,1,None,NaN,NaN,NaN,片手,hon_settei,1,0
1,19-4Q,2016-07-01,3,同営業日,2016-07-05,2,165,292368810,2,パートナー紹介,角田隆,嵯峨優,ミドル企業,1,None,NaN,NaN,NaN,片手,hon_settei,1,1
2,19-4Q,2016-07-01,3,同営業日,2016-07-05,3,664,422112860,3,顧問名鑑登録者,大矢裕,岡大,ミドル企業,1,None,NaN,NaN,NaN,片手,hon_settei,1,2
3,19-4Q,2016-07-01,3,同営業日,2016-07-05,4,455,400652951,4,その他,大仲研,大仲研,ミドル企業,1,None,ミドル企業,1,None,両手,hon_settei,1,3
4,19-4Q,2016-07-01,3,同営業日,2016-07-05,5,665,422112860,5,パートナー紹介,角田隆,岡大,ミドル企業,1,None,NaN,NaN,NaN,片手,hon_settei,1,4


In [24]:
# ==========================================
# 1. あなたが並べたい理想の順番を「リスト」で定義します
# 今後KPIが増えたら、ここにどんどんカンマ区切りで追記していくだけでOKです！
# ==========================================
type_order = [
    'hon_settei',
    'hon_settei_shin',
    'hon_settei_sai',
    'hon_settei_kata',
    'hon_settei_kata_shin',
    'hon_settei_kata_sai',
    'hon_settei_ryo',
    'hon_settei_ryo_shin',
    'hon_settei_ryo_sai',
    'hon_settei_sha',
    'hon_settei_nin',
    'kigyo_settei',
    'kigyo_settei_sha',
    'kigyo_settei_nin',

    'hon_settei_han',
    'hon_settei_han_shin',
    'hon_settei_han_sai',
    'hon_settei_han_kata',
    'hon_settei_han_kata_shin',
    'hon_settei_han_kata_sai',
    'hon_settei_han_ryo',
    'hon_settei_han_ryo_shin',
    'hon_settei_han_ryo_sai',
    'hon_settei_han_sha',
    'hon_settei_han_nin',
    'kigyo_settei_han',
    'kigyo_settei_han_sha',
    'kigyo_settei_han_nin'
]

# ==========================================
# 2. hon_combined_data の 'type' カラムに、上で作った「独自の順番（ルール）」を記憶させます
# ==========================================
hon_combined_data['type'] = pd.Categorical(
    hon_combined_data['type'], 
    categories=type_order, 
    ordered=True  # 「この順番に意味があるよ（順序付きだよ）」と教えてあげます
)

In [25]:
#==========================================
# セル5: ピボットテーブルの作成
# ==========================================
# df1: APソース
df1 = hon_combined_data.pivot_table(
    index=["type", "APソース"], columns="Q",
    aggfunc="nunique", values="calc_target"  # ★ここを変更
).fillna(0).reset_index()
df1_fixed = df1.rename(columns={'APソース': '項目名'})
df1_fixed.insert(1, '集計軸', 'APソース')

# df2: 職種 (候担)
df2 = hon_combined_data.pivot_table(
    index=["type", "候担_職種"], columns="Q",
    aggfunc="nunique", values="calc_target"  # ★ここを変更
).fillna(0).reset_index()
df2_fixed = df2.rename(columns={'候担_職種': '項目名'})
df2_fixed.insert(1, '集計軸', '職種')

# df3: レイヤー (候担)
df3 = hon_combined_data.pivot_table(
    index=["type", "候担_レイヤー"], columns="Q",
    aggfunc="nunique", values="calc_target"  # ★ここを変更
).fillna(0).reset_index()
df3_fixed = df3.rename(columns={'候担_レイヤー': '項目名'})
df3_fixed.insert(1, '集計軸', 'レイヤー')

# df4: 担当 (候担)
df4 = hon_combined_data.pivot_table(
    index=["type", "候補者担当"], columns="Q",
    aggfunc="nunique", values="calc_target"  # ★ここを変更
).fillna(0).reset_index()

valid_members = master2['sei_plus'].dropna().unique()
df4 = df4[df4['候補者担当'].isin(valid_members)].copy()
df4_fixed = df4.rename(columns={'候補者担当': '項目名'})
df4_fixed.insert(1, '集計軸', '担当')

# df5: 企担_職種
df5 = hon_combined_data.pivot_table(
    index=["type", "企担_職種"], columns="Q",
    aggfunc="nunique", values="calc_target"
).fillna(0).reset_index()
df5_fixed = df5.rename(columns={'企担_職種': '項目名'})
df5_fixed.insert(1, '集計軸', '企担_職種')

# df6: 企業担当
df6 = hon_combined_data.pivot_table(
    index=["type", "企業担当"], columns="Q",
    aggfunc="nunique", values="calc_target"
).fillna(0).reset_index()
# ※企業担当も同じ master2 の名簿で絞り込みます
df6 = df6[df6['企業担当'].isin(valid_members)].copy()
df6_fixed = df6.rename(columns={'企業担当': '項目名'})
df6_fixed.insert(1, '集計軸', '企業担当')


# ==========================================
# 縦にガッチャンコ（結合）
# ==========================================
all_pivot_df = pd.concat([df1_fixed, df2_fixed, df3_fixed, df4_fixed, df5_fixed, df6_fixed], ignore_index=True)

# 1. 'type' 列を並び替え用のカテゴリ型に変換（空欄があってもエラーにならないようにします）
all_pivot_df['type'] = pd.Categorical(all_pivot_df['type'], categories=type_order, ordered=True)

# 2. 数値が入るべき列（Qの列）だけを特定して、空欄を 0 で埋めます
# type, 集計軸, 項目名 以外の列が対象になります
numeric_cols = all_pivot_df.columns.difference(['type', '集計軸', '項目名'])
all_pivot_df = all_pivot_df[all_pivot_df[numeric_cols].sum(axis=1) > 0]

# 3. 独自の順番通りに並び替えを実行
all_pivot_df = all_pivot_df.sort_values(by=['type', '集計軸', '項目名'])

# 4. 見た目を整える
all_pivot_df.columns.name = None

# 本交渉の結果を別名で保存しておきます
hon_pivot_df = all_pivot_df.copy()

# 確認表示
display(hon_pivot_df.head())
display(hon_pivot_df.tail())

C:\Users\suehara\AppData\Local\Temp\ipykernel_19584\3420421046.py:5: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  df1 = hon_combined_data.pivot_table(
C:\Users\suehara\AppData\Local\Temp\ipykernel_19584\3420421046.py:13: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  df2 = hon_combined_data.pivot_table(
C:\Users\suehara\AppData\Local\Temp\ipykernel_19584\3420421046.py:21: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  df3 = hon_combined_data.pivot_table(
C:\Users\suehara\AppData\Local\Temp\ipykernel_1958

,type,集計軸,項目名,19-1Q,19-4Q,20-1Q,20-2Q,20-3Q,20-4Q,21-1Q,21-2Q,21-3Q,21-4Q,22-1Q,22-2Q,22-3Q,22-4Q,23-1Q,23-2Q,23-3Q,23-4Q,24-1Q,24-2Q,24-3Q,24-4Q,25-1Q,25-2Q,25-3Q,25-4Q,26-1Q,26-2Q,26-3Q,26-4Q,27-1Q,27-2Q,27-3Q,27-4Q,28-1Q,28-2Q,28-3Q,28-4Q,29-1Q,29-2Q,29-3Q
0,hon_settei,APソース,SMAP,0,63,57,34,44,34,37,27,21,34,52,73,94,84,99,76,68,97,93,135,115,133,93,88,105,124,234,250,347,335,373,343,449,463,424,478,483,533,521,542,232
1,hon_settei,APソース,その他,0,13,45,36,48,29,38,38,23,31,30,19,33,41,46,51,53,64,64,76,74,51,52,51,33,43,29,28,33,38,33,28,43,28,35,48,46,59,40,38,21
2,hon_settei,APソース,パートナー紹介,0,110,123,152,158,153,135,122,145,179,159,176,178,135,166,148,93,139,150,203,152,130,83,74,71,107,80,50,67,96,91,145,137,149,116,120,79,95,68,101,41
3,hon_settei,APソース,人事部紹介,1,38,25,31,38,20,25,30,17,54,32,68,43,49,60,48,47,48,57,100,110,75,49,69,54,51,40,32,38,35,30,35,23,28,26,21,21,28,14,27,7
4,hon_settei,APソース,転機,0,23,35,62,134,118,142,172,196,206,191,262,322,369,428,380,385,404,404,387,423,313,295,275,206,272,183,140,207,267,262,212,280,247,307,336,272,284,252,219,94


,type,集計軸,項目名,19-1Q,19-4Q,20-1Q,20-2Q,20-3Q,20-4Q,21-1Q,21-2Q,21-3Q,21-4Q,22-1Q,22-2Q,22-3Q,22-4Q,23-1Q,23-2Q,23-3Q,23-4Q,24-1Q,24-2Q,24-3Q,24-4Q,25-1Q,25-2Q,25-3Q,25-4Q,26-1Q,26-2Q,26-3Q,26-4Q,27-1Q,27-2Q,27-3Q,27-4Q,28-1Q,28-2Q,28-3Q,28-4Q,29-1Q,29-2Q,29-3Q
412,kigyo_settei_han_nin,職種,SMAP,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
414,kigyo_settei_han_nin,職種,フロント,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,3,2,3,1
415,kigyo_settei_han_nin,職種,ミドル企業,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,7,8,6,3,1
417,kigyo_settei_han_nin,職種,ミドル候B,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6,5,5,8,0
418,kigyo_settei_han_nin,職種,半常勤,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5,15,11,15,10


In [26]:
# ==========================================
# 最終工程: データの統合とスプレッドシート書き出し
# ==========================================

shoki_pivot_df['type'] = shoki_pivot_df['type'].astype(str)
hon_pivot_df['type'] = hon_pivot_df['type'].astype(str)

# 1. 初期交渉と本交渉のデータを縦にガッチャンコ！
final_all_df = pd.concat([shoki_pivot_df, hon_pivot_df], ignore_index=True)


# 2. 全KPI（type）の並び順リストを定義
full_type_order = [
    'shoki_settei', 'shoki_jisshi',
    'hon_settei', 'hon_jisshi',
    'hon_settei_shin', 'hon_settei_sai',
    'hon_settei_kata', 'hon_settei_kata_shin', 'hon_settei_kata_sai',
    'hon_settei_ryo', 'hon_settei_ryo_shin', 'hon_settei_ryo_sai',
    'hon_settei_sha','hon_settei_nin','kigyo_settei','kigyo_settei_sha','kigyo_settei_nin',

    'hon_settei_han',
    'hon_settei_han_shin','hon_settei_han_sai',
    'hon_settei_han_kata','hon_settei_han_kata_shin','hon_settei_han_kata_sai',
    'hon_settei_han_ryo','hon_settei_han_ryo_shin','hon_settei_han_ryo_sai',
    'hon_settei_han_sha','hon_settei_han_nin',
    'kigyo_settei_han','kigyo_settei_han_sha','kigyo_settei_han_nin'
]

# 3. 改めて並び替え（ソート）を適用
final_all_df['type'] = pd.Categorical(final_all_df['type'], categories=full_type_order, ordered=True)


# 数値列（Qの列）の空欄を 0 で埋める
numeric_cols = final_all_df.columns.difference(['type', '集計軸', '項目名'])
final_all_df[numeric_cols] = final_all_df[numeric_cols].fillna(0)

# 並び替え実行
final_all_df = final_all_df.sort_values(by=['type', '集計軸', '項目名'])
final_all_df.columns.name = None




In [27]:

display(final_all_df.tail())

,type,集計軸,項目名,18-1Q,18-2Q,18-3Q,18-4Q,19-1Q,19-2Q,19-3Q,19-4Q,20-1Q,20-2Q,20-3Q,20-4Q,21-1Q,21-2Q,21-3Q,21-4Q,22-1Q,22-2Q,22-3Q,22-4Q,23-1Q,23-2Q,23-3Q,23-4Q,24-1Q,24-2Q,24-3Q,24-4Q,25-1Q,25-2Q,25-3Q,25-4Q,26-1Q,26-2Q,26-3Q,26-4Q,27-1Q,27-2Q,27-3Q,27-4Q,28-1Q,28-2Q,28-3Q,28-4Q,29-1Q,29-2Q,29-3Q
6203,kigyo_settei_han_nin,職種,SMAP,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
6204,kigyo_settei_han_nin,職種,フロント,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,3,2,3,1
6205,kigyo_settei_han_nin,職種,ミドル企業,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,7,8,6,3,1
6206,kigyo_settei_han_nin,職種,ミドル候B,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6,5,5,8,0
6207,kigyo_settei_han_nin,職種,半常勤,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5,15,11,15,10


In [28]:



# ★追加の安全対策★ (修正版)
# 並び替え（ソート）はもう完了したので、type列の厳格なカテゴリ型ルールを解除し、普通の「文字型」に戻します
final_all_df['type'] = final_all_df['type'].astype(str)

# これで表全体に対して安全に NaN を 空文字("") に変換できます
final_all_df = final_all_df.fillna("")

# 4. スプレッドシートへの書き出し処理
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly',
          'https://www.googleapis.com/auth/spreadsheets']
json_path = r"Z:\Users\suehara\Documents\python\analysis\yojitu\python_ss\credentials.json"
service = ps.get_auth(SCOPES, json_path)

OUTPUT_SPREADSHEET_ID = '12ws4AVPj6Xu7t0oTDYAmaTN_JpAVCPgk9aYTemJcIl4'
Sheet_NAME = 'pivot'

# 書き出し用データの準備
data_to_export = [final_all_df.columns.values.tolist()] + final_all_df.values.tolist()

# シートをクリアしてから一気に書き込み
service.spreadsheets().values().clear(spreadsheetId=OUTPUT_SPREADSHEET_ID, range=Sheet_NAME).execute()
ps.update_ss(OUTPUT_SPREADSHEET_ID, Sheet_NAME + '!A1', data_to_export, service)

print("✨ 初期交渉・本交渉すべてのデータを統合してスプレッドシートへの出力が完了しました！")
display(final_all_df.head(5)) 
display(final_all_df.tail(5)) 

✨ 初期交渉・本交渉すべてのデータを統合してスプレッドシートへの出力が完了しました！


,type,集計軸,項目名,18-1Q,18-2Q,18-3Q,18-4Q,19-1Q,19-2Q,19-3Q,19-4Q,20-1Q,20-2Q,20-3Q,20-4Q,21-1Q,21-2Q,21-3Q,21-4Q,22-1Q,22-2Q,22-3Q,22-4Q,23-1Q,23-2Q,23-3Q,23-4Q,24-1Q,24-2Q,24-3Q,24-4Q,25-1Q,25-2Q,25-3Q,25-4Q,26-1Q,26-2Q,26-3Q,26-4Q,27-1Q,27-2Q,27-3Q,27-4Q,28-1Q,28-2Q,28-3Q,28-4Q,29-1Q,29-2Q,29-3Q
0,shoki_settei,APソース,SMAP,0.0,2.0,0.0,0.0,0,10.0,43.0,17,36,58,40,41,36,50,42,76,71,97,112,101,106,96,180,144,231,265,185,334,183,184,203,508,876,1173,928,930,1128,1192,1282,1414,1429,1427,1545,1568,1622,1577,621
1,shoki_settei,APソース,その他,0.0,0.0,0.0,1.0,1,11.0,26.0,30,57,118,68,142,77,132,115,106,76,64,61,72,65,72,83,72,96,114,87,59,64,64,48,66,54,27,45,44,62,85,82,92,79,95,105,132,83,91,24
2,shoki_settei,APソース,パートナー紹介,1.0,1.0,0.0,0.0,11,68.0,90.0,36,70,171,151,247,252,202,368,294,324,269,164,201,236,190,249,183,315,426,377,340,203,173,221,225,172,175,122,156,149,254,276,230,149,143,119,140,147,137,52
3,shoki_settei,APソース,人事部,0.0,0.0,0.0,0.0,2,17.0,13.0,22,25,39,33,34,30,57,41,72,57,89,53,91,97,78,81,86,125,126,100,79,99,88,62,77,102,109,69,57,82,73,56,81,85,70,51,61,108,162,21
4,shoki_settei,APソース,解放顧問,1.0,0.0,0.0,0.0,0,2.0,21.0,3,18,24,36,50,82,117,87,66,37,24,32,25,61,33,90,20,152,89,60,79,65,9,48,79,105,40,98,84,54,107,55,43,24,27,28,37,31,10,3


,type,集計軸,項目名,18-1Q,18-2Q,18-3Q,18-4Q,19-1Q,19-2Q,19-3Q,19-4Q,20-1Q,20-2Q,20-3Q,20-4Q,21-1Q,21-2Q,21-3Q,21-4Q,22-1Q,22-2Q,22-3Q,22-4Q,23-1Q,23-2Q,23-3Q,23-4Q,24-1Q,24-2Q,24-3Q,24-4Q,25-1Q,25-2Q,25-3Q,25-4Q,26-1Q,26-2Q,26-3Q,26-4Q,27-1Q,27-2Q,27-3Q,27-4Q,28-1Q,28-2Q,28-3Q,28-4Q,29-1Q,29-2Q,29-3Q
6203,kigyo_settei_han_nin,職種,SMAP,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
6204,kigyo_settei_han_nin,職種,フロント,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,3,2,3,1
6205,kigyo_settei_han_nin,職種,ミドル企業,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,7,8,6,3,1
6206,kigyo_settei_han_nin,職種,ミドル候B,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6,5,5,8,0
6207,kigyo_settei_han_nin,職種,半常勤,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5,15,11,15,10


In [29]:
# ==========================================
# NaNの原因（リストから漏れている名前）を特定するコード
# ==========================================

# 1. 念のため、初期交渉と本交渉のすべての type を「文字」として取得します
unique_shoki_types = shoki_pivot_df['type'].astype(str).unique()
unique_hon_types = hon_pivot_df['type'].astype(str).unique()

# 2. それらをガッチャンコして、存在しているすべての type 名のリストを作ります
all_existing_types = list(set(unique_shoki_types) | set(unique_hon_types))

# 3. リストに登録した「full_type_order」と比較して、漏れている犯人を探します
full_type_order = [
    'shoki_settei', 'shoki_jisshi',
    'hon_settei', 'hon_jisshi',
    'hon_settei_shin', 'hon_settei_sai',
    'hon_settei_kata', 'hon_settei_kata_shin', 'hon_settei_kata_sai',
    'hon_settei_ryo', 'hon_settei_ryo_shin', 'hon_settei_ryo_sai',
    'hon_settei_sha','hon_settei_nin','kigyo_settei','kigyo_settei_sha','kigyo_settei_nin',
    
    'hon_settei_han',
    'hon_settei_han_shin','hon_settei_han_sai',
    'hon_settei_han_kata','hon_settei_han_kata_shin','hon_settei_han_kata_sai',
    'hon_settei_han_ryo','hon_settei_han_ryo_shin','hon_settei_han_ryo_sai',
    'hon_settei_han_sha','hon_settei_han_nin',
    'kigyo_settei_han','kigyo_settei_han_sha','kigyo_settei_han_nin'
]

# 犯人（実際のデータには存在するのに、full_type_order に書いていない名前）をあぶり出します
missing_types = [t for t in all_existing_types if t not in full_type_order]

print("🚨 リストに書き忘れている type 名は以下の通りです：")
print(missing_types)

🚨 リストに書き忘れている type 名は以下の通りです：
[]


## 成約数と顧客支持ポイント

In [30]:
# ==========================================
# 成約数情報の取得（必要な列のみに絞り込み）
# ==========================================

# 1. 取得範囲を AN列まで広げて取得
SPREADSHEET_ID = "1ITzx2eIAMpiepjGVrDSf-FR1XcAKzYJMqFZuS6-8qb0"
Sheet_NAME = 'data!'
Sheet_row = "A:AN" # AN列まで取得するように変更
RANGE_NAME = Sheet_NAME + Sheet_row

# 全データを一旦取得
yomi_full = ps.get_ss(SPREADSHEET_ID, RANGE_NAME, service)

# 2. 必要な列だけを抽出（列名、または列番号で指定）
# ※ps.get_ss が 1行目をヘッダーとして読み込んでいる前提です
target_cols = [
    '案件id（RZ）', '月', '案件No（SC）', '計上日', 'クライアント正式名称', 
    '候補者', '売上種別（商品内容）', '基準年収', '報酬率', 
    '営業売上合計', '所属課', '氏名', '顧客支持ポイント', '貢献引当後pt', 
    '担当', '内定数フラグ'
]

# 存在する列だけを安全に抽出
yomi_old = yomi_full[target_cols].copy()

# 確認表示
display(yomi_old.tail())

,案件id（RZ）,月,案件No（SC）,計上日,クライアント正式名称,候補者,売上種別（商品内容）,基準年収,報酬率,営業売上合計,所属課,氏名,顧客支持ポイント,貢献引当後pt,担当,内定数フラグ
62517,,4,,2026/04/30,Anique株式会社,,顧問名鑑,,,,ロンザン,高山莉,31.9846,28.7861,None,None
62518,,4,,2026/04/30,Anique株式会社,,顧問名鑑,,,,ロンザン,ロンザンその他,21.3231,19.1908,None,None
62519,,5,,2026/05/01,株式会社オークウェーブ,,ビジネスタンク,,,,ロンザン,侭田ゆ,52.8682,47.5813,None,None
62520,,5,,2026/05/01,株式会社ビジョン・コンサルティング,,顧問名鑑,,,,ロンザン,向井実,52.5312,47.2781,None,None
62521,,5,,2026/05/01,株式会社ビジョン・コンサルティング,,顧問名鑑,,,,ロンザン,ロンザンその他,14.592,13.1328,None,None


In [31]:
# ==========================================
# 今Qのヨミ表を取得（特定のカラムのみ抽出）
# ==========================================

# 1. 取得範囲の設定（AZ列まで広めに取得）
SPREADSHEET_ID = "1yZdENO78p8AkftwaNlwNqP_NQVwKfPYYpjBGjUccflo"
Sheet_NAME = 'ヨミ表!'
Sheet_row = "A10:AZ" # 10行目からAZ列まで
RANGE_NAME = Sheet_NAME + Sheet_row

# 一旦すべてのデータを取得
yomi_nowQ_full = ps.get_ss(SPREADSHEET_ID, RANGE_NAME, service)

# ------------------------------------------
# ★追加：カラム名のクレンジング（ノイズ除去）
# ------------------------------------------
# \s+ は「スペースや改行、タブなどの空白文字」を表します。これを空文字('')に置き換えます。
yomi_nowQ_full.columns = [re.sub(r'\s+', '', str(col)) for col in yomi_nowQ_full.columns]

# 2. 抽出したいカラムのリスト（過去ヨミ表と共通）
target_cols = [
    '案件id（RZ）', '月', '案件No（SC）', '計上日', 'クライアント正式名称', 
    '候補者', '売上種別（商品内容）', '基準年収', '報酬率', 
    '営業売上合計', '所属課', '氏名', '顧客支持ポイント', '貢献引当後pt', 
    '担当', '内定数フラグ'
]

# 3. 指定したカラムだけを抽出し、メモリ効率のためにコピーを作成
# ※列名が存在しない場合のエラーを防ぐため、実際の列に含まれるものだけを抽出します
yomi_nowQ = yomi_nowQ_full[[c for c in target_cols if c in yomi_nowQ_full.columns]].copy()

# 確認表示
display(yomi_nowQ.head())

,案件id（RZ）,月,案件No（SC）,計上日,クライアント正式名称,候補者,売上種別（商品内容）,基準年収,報酬率,営業売上合計,所属課,氏名,顧客支持ポイント,貢献引当後pt,内定数フラグ
1,28413,2026年4月,20402,2026/4/7,株式会社サンマルクホールディングス,飛田 氏,シニアスカウト報酬,900,69%,621.0000,シニア引当,CXL引当,62.1000,55.8900,
2,28413,2026年4月,20402,2026/4/7,株式会社サンマルクホールディングス,飛田 氏,シニアスカウト報酬,900,69%,621.0000,シニア引当,サービス引当,55.8900,50.3010,
3,28413,2026年4月,20402,2026/4/7,株式会社サンマルクホールディングス,飛田 氏,シニアスカウト報酬,900,69%,621.0000,商材間調整,商材間調整,100.6020,90.5418,
4,28413,2026年4月,20402,2026/4/7,株式会社サンマルクホールディングス,飛田 氏,シニアスカウト報酬,900,69%,621.0000,外部原価,外部原価,120.7220,108.6498,
5,28413,2026年4月,20402,2026/4/7,株式会社サンマルクホールディングス,飛田 氏,シニアスカウト報酬,900,69%,621.0000,ロンザン,RZ優良企業開拓引当,8.0280,7.2252,


In [32]:
yomi_nowQ.columns

Index(['案件id（RZ）', '月', '案件No（SC）', '計上日', 'クライアント正式名称', '候補者', '売上種別（商品内容）',
       '基準年収', '報酬率', '営業売上合計', '所属課', '氏名', '顧客支持ポイント', '貢献引当後pt', '内定数フラグ'],
      dtype='object')

In [33]:
# ==========================================
# ヨミ表データの縦結合（過去分 + 今Q分）
# ==========================================

# pd.concat を使って縦に結合します
# ignore_index=True を指定することで、インデックス（行番号）を 0 から振り直してキレイにします
yomi_all = pd.concat([yomi_old, yomi_nowQ], ignore_index=True)

yomi_all.loc[yomi_all['氏名'].isin(['京谷悠子', '京谷悠']), '氏名'] = 'yuko-kyotani'
yomi_all.loc[yomi_all['氏名'] == '竹下綾', '氏名'] = 'aya-takeshita'

# 結合結果の確認
print(f"過去データの行数: {len(yomi_old)}")
print(f"今Qデータの行数 : {len(yomi_nowQ)}")
print(f"結合後の合計行数: {len(yomi_all)}")

# データの先頭と末尾を表示して、正しく結合されているか確認
display(yomi_all.head())
display(yomi_all.tail())
display(yomi_all.columns)

過去データの行数: 62521
今Qデータの行数 : 26939
結合後の合計行数: 89460


,案件id（RZ）,月,案件No（SC）,計上日,クライアント正式名称,候補者,売上種別（商品内容）,基準年収,報酬率,営業売上合計,所属課,氏名,顧客支持ポイント,貢献引当後pt,担当,内定数フラグ
0,,4,8606,2016/04/15,株式会社フジダン,白川 正明 氏,スカウト報酬（通常）,630,0.55,,ロンザン,宮崎佳,82.467,82.467,None,None
1,,4,,2016/04/15,株式会社林間,岩本 昌和 氏,スカウト報酬（通常）,740,0.58,,ロンザン,宮崎佳,102.1496,102.1496,None,None
2,内定内定71,4,,2016/04/21,株式会社星野産商,鼠入 宏明 氏,シニアスカウト報酬,1110,0.58,,シニア引当,サービス引当,64.38,64.38,,None
3,内定内定71,4,,2016/04/21,株式会社星野産商,鼠入 宏明 氏,シニアスカウト報酬,1110,0.58,,シニア引当,CXL引当,128.76,128.76,,None
4,内定内定71,4,,2016/04/21,株式会社星野産商,鼠入 宏明 氏,シニアスカウト報酬,1110,0.58,,ロンザン,長崎文,191.5305,191.5305,候補者担当,1


,案件id（RZ）,月,案件No（SC）,計上日,クライアント正式名称,候補者,売上種別（商品内容）,基準年収,報酬率,営業売上合計,所属課,氏名,顧客支持ポイント,貢献引当後pt,担当,内定数フラグ
89455,,,None,,,,,,,,,,,0.0000,NaN,None
89456,,,None,,,,,,,,,,,0.0000,NaN,None
89457,,,None,,,,,,,,,,,0.0000,NaN,None
89458,,,None,,,,,,,,,,,0.0000,NaN,None
89459,,,None,,,,,,,,,,,0.0000,NaN,None


Index(['案件id（RZ）', '月', '案件No（SC）', '計上日', 'クライアント正式名称', '候補者', '売上種別（商品内容）',
       '基準年収', '報酬率', '営業売上合計', '所属課', '氏名', '顧客支持ポイント', '貢献引当後pt', '担当',
       '内定数フラグ'],
      dtype='object')

In [34]:
# ==========================================
# yomi_all の前処理とマスタマージ
# ==========================================

# 1. データ型の変換
# '計上日' を日付型に変換
yomi_all['計上日'] = pd.to_datetime(yomi_all['計上日'], errors='coerce')

# それ以外の列を object 型に一括変換
other_cols = yomi_all.columns.difference(['計上日'])
yomi_all[other_cols] = yomi_all[other_cols].astype('object')


# 2. Q_masterのマージ（計上日基準）
yomi_all = (
    yomi_all.merge(q_master_subset, left_on='計上日', right_on='日付', how='left')
    .drop(columns=['日付'])
    .rename(columns={
        'Q': 'keijo_Q', 
        '月': 'keijo_月', 
        '営業日': 'keijo_営業日', 
        '同営業日比較': 'keijo_同営業日比較'
    })
)


In [35]:
# 3. honkosho_data から「担当者属性マスター」を作成
# 案件IDに紐づく担当者や属性は基本的に1つのため、重複を排除してマスター化します
hon_tanto_master = honkosho_data[[
    'anken_id', 'kohosha_id', 'APsource', 
    'kohosha_tanto', 'kigyo_tanto', 
    '候担_職種', '企担_職種', 
    '候担_レイヤー', '企担_レイヤー', 
    '候担_ロンザン所属フラグ', '企担_ロンザン所属フラグ', '組み手', 'hanjokin'
]].drop_duplicates('anken_id')


In [36]:
# ==========================================
# マージ前の徹底洗浄（ここが重要！）
# ==========================================

# 1. hon_tanto_master のキーを洗浄
hon_tanto_master['anken_id'] = hon_tanto_master['anken_id'].astype(str).str.strip()

# 2. yomi_all のカラム名を扱いやすくリネーム（改行コード対策）
yomi_all = yomi_all.rename(columns={c: '案件id_RZ' for c in yomi_all.columns if '案件id' in c})
yomi_all = yomi_all.rename(columns={c: '案件No_SC' for c in yomi_all.columns if '案件' in c and 'No' in c})

# 3. yomi_all のキーを洗浄（数値型を文字列に変え、空白を消す）
yomi_all['案件id_RZ'] = yomi_all['案件id_RZ'].astype(str).str.strip()

# 4. すでに yomi_all に存在する「マージで持ってきたい列」を一旦削除（重複防止）
# これをしないと _x _y という列が増えるだけで中身が NaN のままに見えます
cols_to_drop = [
    'kohosha_id', 'APsource', 'kohosha_tanto', 'kigyo_tanto', 
    '候担_職種', '企担_職種', '候担_レイヤー', '企担_レイヤー', 
    '候担_ロンザン所属フラグ', '企担_ロンザン所属フラグ', '組み手'
]
yomi_all = yomi_all.drop(columns=[c for c in cols_to_drop if c in yomi_all.columns])

# ==========================================
# いざ、再マージ！
# ==========================================
yomi_all = yomi_all.merge(
    hon_tanto_master, 
    left_on='案件id_RZ', 
    right_on='anken_id', 
    how='left'
)

# 洗浄のために作った temporary な列を消してスッキリ
if 'anken_id' in yomi_all.columns:
    yomi_all = yomi_all.drop(columns=['anken_id'])



In [37]:

# 結果確認
print("紐付け成功数（NaNでない数）:", yomi_all['kohosha_id'].notna().sum())
display(yomi_all.sample(min(10, len(yomi_all))))

# ==========================================
# ローデータ（yomi_all）のスプレッドシート出力
# ==========================================

# 1. 出力用データの準備
output_raw_df = yomi_all.copy()

# 【エラー対策】全ての datetime/Timestamp 型の列を文字列に変換します
# これにより JSON serializable な形式になります
for col in output_raw_df.select_dtypes(include=['datetime', 'datetime64']).columns:
    # 日付形式（YYYY-MM-DD）の文字列に変換。時間が不要な場合は strftime('%Y-%m-%d')
    output_raw_df[col] = output_raw_df[col].dt.strftime('%Y-%m-%d')

# 欠損値を空文字に変換（NaT も上の処理で NaN になっている場合はここで空文字になります）
output_raw_df = output_raw_df.fillna("")

# 2. DataFrameをヘッダー付きのリスト形式に変換
header = output_raw_df.columns.tolist()
values_to_send = [header] + output_raw_df.values.tolist()

# 3. 書き出し先の設定
RAW_SPREADSHEET_ID = "12ws4AVPj6Xu7t0oTDYAmaTN_JpAVCPgk9aYTemJcIl4"
RAW_SHEET_NAME = "raw_seiyaku"

print(f"「{RAW_SHEET_NAME}」シートへの書き出しを開始します...")

# 4. 指定したシートのA1セルから書き出しを実行
# 既存のヘルパー関数 ps.update_ss を使用します
ps.update_ss(RAW_SPREADSHEET_ID, f"{RAW_SHEET_NAME}!A1", values_to_send, service)

print(f"✅ スプレッドシートへの出力が完了しました！")

紐付け成功数（NaNでない数）: 54041


,案件id_RZ,月_x,案件No_SC,計上日,クライアント正式名称,候補者,売上種別（商品内容）,基準年収,報酬率,営業売上合計,所属課,氏名,顧客支持ポイント,貢献引当後pt,担当,内定数フラグ,keijo_Q,月_y,keijo_営業日,keijo_同営業日比較,kohosha_id,APsource,kohosha_tanto,kigyo_tanto,候担_職種,企担_職種,候担_レイヤー,企担_レイヤー,候担_ロンザン所属フラグ,企担_ロンザン所属フラグ,組み手,hanjokin
4867,3315,9,,2018-09-28,株式会社ＳＴＩフードホールディングス,番 英一 氏,シニアスカウト報酬,1200,0.58,696,外部原価,外部原価,73.08,73.08,外部原価①,None,21-4Q,2018-09-01,59,,7749,パートナー紹介,長松大,片郁,,NaN,None,NaN,1,NaN,片手,
65496,,,None,NaT,,,,,,,,,,0.0000,NaN,None,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
84982,,,None,NaT,,,,,,,,,,0.0000,NaN,None,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
74826,,,None,NaT,,,,,,,,,,0.0000,NaN,None,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16442,,,,2020-08-31,株式会社プラチナリンク,浦野文男,顧問名鑑,,,,ロンザン,小橋一,18.003,18.003,None,None,23-4Q,2020-08-01,40,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8615,5266,6,新規,2019-06-27,株式会社アフターフィットエンジニアリング,井原 克幸 氏,,2371.3463,0.58,1375.380854,顧問名鑑,竹田康,132.3804072,132.3804072,開拓担当,None,22-3Q,2019-06-01,56,,7711,顧問名鑑登録者,宮崎佳,山口大,,NaN,None,NaN,1,NaN,片手,
32962,16099,4,11718,2023-04-29,金属技研株式会社,横井 伸光 氏,シニアスカウト報酬,915.2576,0.58,530.849408,ロンザン,金子祐,33.28365271,33.28365271,面談担当,None,26-3Q,2023-04-01,18,同営業日,41452,転機,池田温,金子祐,フロント,フロント,022生,既存,1,1,両手,
84438,,,None,NaT,,,,,,,,,,0.0000,NaN,None,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1368,1050,5,,2017-05-29,株式会社シンク・ラボラトリー,横井 克幸 氏,シニアスカウト報酬,1101.285,0.58,,ロンザン,大矢裕,76.0106907,76.0106907,候補者担当②,None,20-3Q,2017-05-01,38,,1264,パートナー紹介,大矢裕,山田あ,,NaN,None,NaN,1,NaN,片手,
7025,4126,3,これから,2019-03-25,株式会社大同機械,春名 一就 氏,シニアスカウト報酬,898.4417,0.58,521.0962,ロンザン,植木大,16.415,16.415,,None,22-2Q,2019-03-01,53,,3596,転機,植木大,植木大,,,None,None,1,1,両手,


「raw_seiyaku」シートへの書き出しを開始します...
✅ スプレッドシートへの出力が完了しました！


In [38]:
# 1. 準備：最新の担当者属性を「計上Q + 氏名」で紐付け直す
# yomi_all の「氏名」と「keijo_Q」を使って、その時点の職種・レイヤーを取得します
yomi_calc_base = yomi_all.copy()

# 紐付け用に master2 を準備（重複削除）
attr_master = master2[['Q', 'sei_plus', '職種', 'レイヤー', 'ロンザン所属フラグ']].drop_duplicates(['Q', 'sei_plus'])

# マージ実行：この「集計用_職種」などを後のピボットで使用します
yomi_calc_base = yomi_calc_base.merge(
    attr_master, 
    left_on=['keijo_Q', '氏名'], 
    right_on=['Q', 'sei_plus'], 
    how='left'
).rename(columns={'職種_y': '集計用_職種', 'レイヤー_y': '集計用_レイヤー', 'ロンザン所属フラグ_y': '集計用_所属フラグ'})

# 2. 抽出用マスクの定義
def is_blank(series):
    s = series.astype(str).str.strip().str.lower()
    return series.isna() | s.isin(['', 'nan', 'none', 'null', '0', '0.0'])




# 3. 各 type ごとのデータ抽出
seiyaku_dfs = []

# --- A. 成約数（従来通り） ---
# 内定数フラグ = 1 （'1' や 1.0 にも対応）
mask_naitei = yomi_calc_base['内定数フラグ'].astype(str).str.strip().str.startswith('1')

# 各種所属フラグ・組み手
mask_koho_rz = yomi_calc_base['候担_ロンザン所属フラグ'].astype(str).str.contains('1')
mask_kigyo_rz = yomi_calc_base['企担_ロンザン所属フラグ'].astype(str).str.contains('1')
mask_kata = yomi_calc_base['組み手'] == '片手'
mask_ryo = yomi_calc_base['組み手'] == '両手'

# ポイント系の条件
mask_ronzan = yomi_calc_base['所属課'].astype(str).str.contains('ロンザン', na=False)
mask_kotei = yomi_calc_base['売上種別（商品内容）'].astype(str).str.contains('固定報酬', na=False)
mask_blank_rz = is_blank(yomi_calc_base['案件id_RZ'])
# mask_blank_sc = is_blank(yomi_calc_base['案件No_SC'])
mask_not_blank = (~mask_blank_rz)

# 半常勤判定マスクを作成
mask_han_yomi = yomi_all['hanjokin'] == '半常勤'
mask_not_han_yomi = ~mask_han_yomi


# ==========================================
# 4. リストに放り込む
# ==========================================
seiyaku_dfs = []

# --- A. 通常_成約数系 ---
seiyaku_dfs.append(yomi_calc_base[mask_naitei & mask_koho_rz & mask_not_han_yomi].assign(type='seiyaku'))
seiyaku_dfs.append(yomi_calc_base[mask_naitei & mask_kata & mask_not_han_yomi].assign(type='seiyaku_kata'))
seiyaku_dfs.append(yomi_calc_base[mask_naitei & mask_ryo & mask_not_han_yomi].assign(type='seiyaku_ryo'))
seiyaku_dfs.append(yomi_calc_base[mask_naitei & mask_kigyo_rz & mask_not_han_yomi].assign(type='kigyo_seiyaku'))

# --- B. 通常_ポイント・売上系 ---
# point: ロンザン & 空白でない & 固定報酬除外
seiyaku_dfs.append(yomi_calc_base[mask_ronzan & mask_not_blank & ~mask_kotei & mask_not_han_yomi].assign(type='point'))
# point_cross: ロンザン & ID/Noが空白
seiyaku_dfs.append(yomi_calc_base[mask_ronzan & mask_blank_rz  & mask_not_han_yomi].assign(type='point_cross'))
# point_kotei: ロンザン & 空白でない & 固定報酬
seiyaku_dfs.append(yomi_calc_base[mask_ronzan & mask_not_blank & mask_kotei & mask_not_han_yomi].assign(type='point_kotei'))

# --- C. 半常勤_成約数系 ---
seiyaku_dfs.append(yomi_calc_base[mask_naitei & mask_koho_rz & mask_han_yomi].assign(type='seiyaku_han'))
seiyaku_dfs.append(yomi_calc_base[mask_naitei & mask_kata & mask_han_yomi].assign(type='seiyaku_han_kata'))
seiyaku_dfs.append(yomi_calc_base[mask_naitei & mask_ryo & mask_han_yomi].assign(type='seiyaku_han_ryo'))
seiyaku_dfs.append(yomi_calc_base[mask_naitei & mask_kigyo_rz & mask_han_yomi].assign(type='kigyo_seiyaku_han'))

# --- B. 半常勤_ポイント・売上系 ---
# point: ロンザン & 空白でない & 固定報酬除外
seiyaku_dfs.append(yomi_calc_base[mask_ronzan & mask_not_blank & ~mask_kotei & mask_han_yomi].assign(type='point_han'))
# point_cross: ロンザン & ID/Noが空白
seiyaku_dfs.append(yomi_calc_base[mask_ronzan & mask_blank_rz & mask_han_yomi].assign(type='point_cross_han'))
# point_kotei: ロンザン & 空白でない & 固定報酬
seiyaku_dfs.append(yomi_calc_base[mask_ronzan & mask_not_blank & mask_kotei & mask_han_yomi].assign(type='point_kotei_han'))


# ==========================================
# 5. まとめて縦にガッチャンコ
# ==========================================
seiyaku_combined_data = pd.concat(seiyaku_dfs, ignore_index=True)

# 顧客支持ポイントと貢献引当後ptを数値化
seiyaku_combined_data['顧客支持ポイント'] = pd.to_numeric(seiyaku_combined_data['顧客支持ポイント'], errors='coerce').fillna(0)
seiyaku_combined_data['貢献引当後pt'] = pd.to_numeric(seiyaku_combined_data['貢献引当後pt'], errors='coerce').fillna(0)

# 結果確認
print("結合後のデータの行数と列数:", seiyaku_combined_data.shape)
print("\n▼ 作成された type ごとの件数")
print(seiyaku_combined_data['type'].value_counts())

結合後のデータの行数と列数: (34531, 38)

▼ 作成された type ごとの件数
type
point                20335
point_cross           5131
seiyaku               3496
seiyaku_kata          1939
kigyo_seiyaku         1641
seiyaku_ryo           1599
point_kotei            186
point_han              134
point_kotei_han         23
seiyaku_han             17
kigyo_seiyaku_han       13
seiyaku_han_ryo         13
seiyaku_han_kata         4
Name: count, dtype: int64


In [39]:
seiyaku_combined_data.columns

Index(['案件id_RZ', '月_x', '案件No_SC', '計上日', 'クライアント正式名称', '候補者', '売上種別（商品内容）',
       '基準年収', '報酬率', '営業売上合計', '所属課', '氏名', '顧客支持ポイント', '貢献引当後pt', '担当',
       '内定数フラグ', 'keijo_Q', '月_y', 'keijo_営業日', 'keijo_同営業日比較', 'kohosha_id',
       'APsource', 'kohosha_tanto', 'kigyo_tanto', '候担_職種', '企担_職種', '候担_レイヤー',
       '企担_レイヤー', '候担_ロンザン所属フラグ', '企担_ロンザン所属フラグ', '組み手', 'hanjokin', 'Q',
       'sei_plus', '職種', 'レイヤー', 'ロンザン所属フラグ', 'type'],
      dtype='object')

In [40]:
# ==========================================
# セル: 成約・ヨミ表データの詳細抽出と属性紐付け
# ==========================================

# 1. 準備：最新の担当者属性を「計上Q + 氏名」で紐付け直す
yomi_calc_base = yomi_all.copy()

# 紐付け用に master2 を準備（重複削除）
# ※ここでは確実に必要なカラムだけを抽出します
attr_master = master2[['Q', 'sei_plus', '職種', 'レイヤー', 'ロンザン所属フラグ']].drop_duplicates(['Q', 'sei_plus'])

# マージ実行：この「集計用_職種」などを後のピボットで使用します
yomi_calc_base = yomi_calc_base.merge(
    attr_master, 
    left_on=['keijo_Q', '氏名'], 
    right_on=['Q', 'sei_plus'], 
    how='left'
).rename(columns={
    '職種': '集計用_職種', 
    'レイヤー': '集計用_レイヤー', 
    'ロンザン所属フラグ': '集計用_所属フラグ'
})
# ★修正ポイント: merge後に付与されるサフィックス（_yなど）に依存せず、元々のカラム名（'職種'など）を直接リネームするように修正しました。

# 2. 抽出用マスクの定義
def is_blank(series):
    s = series.astype(str).str.strip().str.lower()
    return series.isna() | s.isin(['', 'nan', 'none', 'null', '0', '0.0'])

mask_ronzan = yomi_calc_base['所属課'].astype(str).str.contains('ロンザン', na=False)
mask_kotei = yomi_calc_base['売上種別（商品内容）'].astype(str).str.contains('固定報酬', na=False)
mask_blank_rz = is_blank(yomi_calc_base['案件id_RZ'])
# mask_blank_sc = is_blank(yomi_calc_base['案件No_SC'])
mask_not_blank = (~mask_blank_rz) 

# --- A. 成約数系 ---
mask_naitei = yomi_calc_base['内定数フラグ'].astype(str).str.strip().str.startswith('1')
mask_koho_rz = yomi_calc_base['候担_ロンザン所属フラグ'].astype(str).str.contains('1')
mask_kigyo_rz = yomi_calc_base['企担_ロンザン所属フラグ'].astype(str).str.contains('1')
mask_kata = yomi_calc_base['組み手'] == '片手'
mask_ryo = yomi_calc_base['組み手'] == '両手'

# 3. 各 type ごとのデータ抽出とリスト格納
seiyaku_dfs = []

seiyaku_dfs.append(yomi_calc_base[mask_naitei & mask_koho_rz].assign(type='seiyaku'))
seiyaku_dfs.append(yomi_calc_base[mask_naitei & mask_kata].assign(type='seiyaku_kata'))
seiyaku_dfs.append(yomi_calc_base[mask_naitei & mask_ryo].assign(type='seiyaku_ryo'))
seiyaku_dfs.append(yomi_calc_base[mask_naitei & mask_kigyo_rz].assign(type='kigyo_seiyaku'))

# --- B. ポイント・売上系 ---
seiyaku_dfs.append(yomi_calc_base[mask_ronzan & mask_not_blank & ~mask_kotei].assign(type='point'))
seiyaku_dfs.append(yomi_calc_base[mask_ronzan & mask_blank_rz ].assign(type='point_cross'))
seiyaku_dfs.append(yomi_calc_base[mask_ronzan & mask_not_blank & mask_kotei].assign(type='point_kotei'))

# 縦積み
seiyaku_combined_data = pd.concat(seiyaku_dfs, ignore_index=True)

# 顧客支持ポイントと貢献引当後ptを数値化
seiyaku_combined_data['顧客支持ポイント'] = pd.to_numeric(seiyaku_combined_data['顧客支持ポイント'], errors='coerce').fillna(0)
seiyaku_combined_data['貢献引当後pt'] = pd.to_numeric(seiyaku_combined_data['貢献引当後pt'], errors='coerce').fillna(0)

# 結果確認
print("結合後のデータの行数と列数:", seiyaku_combined_data.shape)
print("\n▼ 作成された type ごとの件数")
print(seiyaku_combined_data['type'].value_counts())


# ==========================================
# セル: 成約・ヨミ表のピボットテーブル作成（詳細版）
# ==========================================

pivot_results = []
seiyaku_types = ['seiyaku', 'seiyaku_kata', 'seiyaku_ryo', 'kigyo_seiyaku', 'point', 'point_cross', 'point_kotei']

for t_name in seiyaku_types:
    df_sub = seiyaku_combined_data[seiyaku_combined_data['type'] == t_name].copy()
    if df_sub.empty: continue

    is_point_type = 'point' in t_name

    # ★ typeに応じて、どのカラムを「職種」「レイヤー」「担当」として扱うか切り替える
    if t_name == 'kigyo_seiyaku':
        job_col = '企担_職種'
        layer_col = '企担_レイヤー'
        tanto_col = 'kigyo_tanto'
    elif is_point_type:
        job_col = '集計用_職種'
        layer_col = '集計用_レイヤー'
        tanto_col = '氏名'
    else:  # seiyaku, seiyaku_kata, seiyaku_ryo
        job_col = '候担_職種'
        layer_col = '候担_レイヤー'
        tanto_col = 'kohosha_tanto'

    # --- 1. APソース軸 ---
    val_col_ap = '顧客支持ポイント' if is_point_type else 'value'
    if not is_point_type: df_sub['value'] = 1
    
    p_ap = df_sub.pivot_table(index=['type', 'APsource'], columns='keijo_Q', values=val_col_ap, aggfunc='sum').fillna(0).reset_index()
    pivot_results.append(p_ap.rename(columns={'APsource': '項目名'}).assign(集計軸='APソース'))

    # --- 2. 職種軸 ---
    val_col_job = '貢献引当後pt' if is_point_type else 'value'
    p_job = df_sub.pivot_table(index=['type', job_col], columns='keijo_Q', values=val_col_job, aggfunc='sum').fillna(0).reset_index()
    pivot_results.append(p_job.rename(columns={job_col: '項目名'}).assign(集計軸='職種'))

    # --- 3. レイヤー軸 ---
    p_layer = df_sub.pivot_table(index=['type', layer_col], columns='keijo_Q', values=val_col_job, aggfunc='sum').fillna(0).reset_index()
    pivot_results.append(p_layer.rename(columns={layer_col: '項目名'}).assign(集計軸='レイヤー'))

    # --- 4. 担当軸 ---
    df_tanto = df_sub[df_sub[tanto_col].isin(valid_members)].copy()
    p_tanto = df_tanto.pivot_table(index=['type', tanto_col], columns='keijo_Q', values=val_col_job, aggfunc='sum').fillna(0).reset_index()
    pivot_results.append(p_tanto.rename(columns={tanto_col: '項目名'}).assign(集計軸='担当'))

# 全結合
seiyaku_pivot_df = pd.concat(pivot_results, ignore_index=True)

# ソートとクレンジング
seiyaku_pivot_df['type'] = pd.Categorical(seiyaku_pivot_df['type'], categories=seiyaku_types, ordered=True)

# 0行撲滅（point_crossは残す）
numeric_cols = seiyaku_pivot_df.columns.difference(['type', '集計軸', '項目名'])
seiyaku_pivot_df[numeric_cols] = seiyaku_pivot_df[numeric_cols].fillna(0)
mask_not_zero = seiyaku_pivot_df[numeric_cols].sum(axis=1) > 0
mask_keep_zero = seiyaku_pivot_df['type'] == 'point_cross'
seiyaku_pivot_df = seiyaku_pivot_df[mask_not_zero | mask_keep_zero]

seiyaku_pivot_df = seiyaku_pivot_df.sort_values(by=['type', '集計軸', '項目名'])
display(seiyaku_pivot_df.tail())

結合後のデータの行数と列数: (34531, 38)

▼ 作成された type ごとの件数
type
point            20469
point_cross       5131
seiyaku           3513
seiyaku_kata      1943
kigyo_seiyaku     1654
seiyaku_ryo       1612
point_kotei        209
Name: count, dtype: int64


keijo_Q,type,項目名,19-4Q,20-1Q,20-2Q,20-3Q,20-4Q,21-1Q,21-2Q,21-3Q,21-4Q,22-1Q,22-2Q,22-3Q,22-4Q,23-1Q,23-2Q,23-3Q,23-4Q,24-1Q,24-2Q,24-3Q,24-4Q,25-1Q,25-2Q,25-3Q,25-4Q,26-1Q,26-2Q,26-3Q,26-4Q,27-1Q,27-2Q,27-3Q,27-4Q,28-1Q,28-2Q,28-3Q,28-4Q,29-1Q,29-2Q,29-3Q,集計軸,19-3Q
1170,point_kotei,PDM,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0000,0.0000,0.0000,0.0000,0.0000,58.4820,18.5193,職種,0.0
1171,point_kotei,フロント,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,350.892,210.5352,333.3474,407.0997,1634.3229,970.5588,1483.8217,807.6621,職種,0.0
1172,point_kotei,ミドル企業,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0000,0.0000,174.4713,592.4643,286.8066,785.3710,125.9808,職種,0.0
1173,point_kotei,ミドル候B,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0000,0.0000,0.0000,117.5204,0.0000,0.0000,0.0000,職種,0.0
1174,point_kotei,半常勤,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0000,0.0000,0.0000,0.0000,76.0266,260.9757,70.1784,職種,0.0


In [41]:
# ==========================================
# 最終工程: 全データ（初期・本・成約）の統合と書き出し
# ==========================================

# 1. 各データの type を文字列型に揃える（念のため）
shoki_pivot_df['type'] = shoki_pivot_df['type'].astype(str)
hon_pivot_df['type'] = hon_pivot_df['type'].astype(str)
seiyaku_pivot_df['type'] = seiyaku_pivot_df['type'].astype(str)

# 2. 3つのデータを縦にガッチャンコ！
# 列（Qの列）が多少ズレていても、Pandasが自動で横に広げて統合してくれます
final_all_df = pd.concat([shoki_pivot_df, hon_pivot_df, seiyaku_pivot_df], ignore_index=True)

# 3. 全KPI（type）の最終的な並び順リストを定義
# 本交渉までのリストに、今回追加した成約系の 5つ を末尾に加えます
full_type_order = [
    'shoki_settei', 'shoki_jisshi',
    'hon_settei', 'hon_jisshi',
    'hon_settei_shin', 'hon_settei_sai',
    'hon_settei_kata', 'hon_settei_kata_shin', 'hon_settei_kata_sai',
    'hon_settei_ryo', 'hon_settei_ryo_shin', 'hon_settei_ryo_sai',
    'hon_settei_sha', 'hon_settei_nin',
    'kigyo_settei', 'kigyo_settei_sha', 'kigyo_settei_nin',
    
    'hon_settei_han',
    'hon_settei_han_shin','hon_settei_han_sai',
    'hon_settei_han_kata','hon_settei_han_kata_shin','hon_settei_han_kata_sai',
    'hon_settei_han_ryo','hon_settei_han_ryo_shin','hon_settei_han_ryo_sai',
    'hon_settei_han_sha','hon_settei_han_nin',
    'kigyo_settei_han','kigyo_settei_han_sha','kigyo_settei_han_nin',
    
    'seiyaku','seiyaku_kata','seiyaku_ryo', 'kigyo_seiyaku', 'point', 'point_cross', 'point_kotei' # ← 成約系を追加！
]

# 4. カテゴリ型を適用して、理想の順番でソート
final_all_df['type'] = pd.Categorical(final_all_df['type'], categories=full_type_order, ordered=True)

# 5. 数値列（Qの列）の空欄を 0 で埋める
# 集計軸ごとに時間軸が異なる場合（成約だけ最近しかない等）に発生する NaN を 0 にします
numeric_cols = final_all_df.columns.difference(['type', '集計軸', '項目名'])
final_all_df[numeric_cols] = final_all_df[numeric_cols].fillna(0)

# 並び替え実行（type → 集計軸 → 項目名の順）
final_all_df = final_all_df.sort_values(by=['type', '集計軸', '項目名'])
final_all_df.columns.name = None

# ==========================================
# Googleスプレッドシートへの書き出し
# ==========================================

# スプレッドシートへ送る直前の最終クレンジング
# ① Categorical型はAPIが処理できないため、文字列に戻します
final_all_df['type'] = final_all_df['type'].astype(str)



# ② 書き出し先のシートとIDを指定（※お使いのIDに書き換えてください）
RESULT_SPREADSHEET_ID = "12ws4AVPj6Xu7t0oTDYAmaTN_JpAVCPgk9aYTemJcIl4" # 出力先のIDを確認してください
SHEET_NAME = 'pivot' # 書き出し先のシート名

# ③ APIエラー防止：すべての NaN を空文字 "" に変えてから出力
# これをしないと、数値列以外の NaN が原因で HttpError 400 が出ることがあります
output_df = final_all_df.fillna("")

# ★ここが重要！ DataFrameをリスト形式に変換します
# 1. ヘッダー（列名）を取得
header = output_df.columns.tolist()
# 2. データ本体をリストに変換し、ヘッダーと合体させる
values_to_send = [header] + output_df.values.tolist()

In [42]:

print("最終データの書き出しを開始します...")
ps.update_ss(RESULT_SPREADSHEET_ID, f"{SHEET_NAME}!A1", values_to_send, service)

print("✅ 全データの統合とスプレッドシートへの出力が完了しました！")
display(output_df.head())

最終データの書き出しを開始します...
✅ 全データの統合とスプレッドシートへの出力が完了しました！


,type,集計軸,項目名,18-1Q,18-2Q,18-3Q,18-4Q,19-1Q,19-2Q,19-3Q,19-4Q,20-1Q,20-2Q,20-3Q,20-4Q,21-1Q,21-2Q,21-3Q,21-4Q,22-1Q,22-2Q,22-3Q,22-4Q,23-1Q,23-2Q,23-3Q,23-4Q,24-1Q,24-2Q,24-3Q,24-4Q,25-1Q,25-2Q,25-3Q,25-4Q,26-1Q,26-2Q,26-3Q,26-4Q,27-1Q,27-2Q,27-3Q,27-4Q,28-1Q,28-2Q,28-3Q,28-4Q,29-1Q,29-2Q,29-3Q
0,shoki_settei,APソース,SMAP,0.0,2.0,0.0,0.0,0.0,10.0,43.0,17.0,36.0,58.0,40.0,41.0,36.0,50.0,42.0,76.0,71.0,97.0,112.0,101.0,106.0,96.0,180.0,144.0,231.0,265.0,185.0,334.0,183.0,184.0,203.0,508.0,876.0,1173.0,928.0,930.0,1128.0,1192.0,1282.0,1414.0,1429.0,1427.0,1545.0,1568.0,1622.0,1577.0,621.0
1,shoki_settei,APソース,その他,0.0,0.0,0.0,1.0,1.0,11.0,26.0,30.0,57.0,118.0,68.0,142.0,77.0,132.0,115.0,106.0,76.0,64.0,61.0,72.0,65.0,72.0,83.0,72.0,96.0,114.0,87.0,59.0,64.0,64.0,48.0,66.0,54.0,27.0,45.0,44.0,62.0,85.0,82.0,92.0,79.0,95.0,105.0,132.0,83.0,91.0,24.0
2,shoki_settei,APソース,パートナー紹介,1.0,1.0,0.0,0.0,11.0,68.0,90.0,36.0,70.0,171.0,151.0,247.0,252.0,202.0,368.0,294.0,324.0,269.0,164.0,201.0,236.0,190.0,249.0,183.0,315.0,426.0,377.0,340.0,203.0,173.0,221.0,225.0,172.0,175.0,122.0,156.0,149.0,254.0,276.0,230.0,149.0,143.0,119.0,140.0,147.0,137.0,52.0
3,shoki_settei,APソース,人事部,0.0,0.0,0.0,0.0,2.0,17.0,13.0,22.0,25.0,39.0,33.0,34.0,30.0,57.0,41.0,72.0,57.0,89.0,53.0,91.0,97.0,78.0,81.0,86.0,125.0,126.0,100.0,79.0,99.0,88.0,62.0,77.0,102.0,109.0,69.0,57.0,82.0,73.0,56.0,81.0,85.0,70.0,51.0,61.0,108.0,162.0,21.0
4,shoki_settei,APソース,解放顧問,1.0,0.0,0.0,0.0,0.0,2.0,21.0,3.0,18.0,24.0,36.0,50.0,82.0,117.0,87.0,66.0,37.0,24.0,32.0,25.0,61.0,33.0,90.0,20.0,152.0,89.0,60.0,79.0,65.0,9.0,48.0,79.0,105.0,40.0,98.0,84.0,54.0,107.0,55.0,43.0,24.0,27.0,28.0,37.0,31.0,10.0,3.0
